In [1]:
import sys

from pathlib import Path

print(sys.executable)

print(Path.cwd())

c:\dev\llm-data-analysis-course\.venv\Scripts\python.exe
c:\dev\llm-data-analysis-course\notebooks\ch09


### STEP 1. 예측 문제와 예측 시점을 먼저 정의합니다
 
무엇을 예측하는가?

예측 단위는 무엇인가?

언제 예측하는가?

그 시점에 실제로 알 수 있는 정보는 무엇인가?

### STEP 2. Feature Leakage Audit을 수행합니다

Feature Leakage Audit(특성 유출 점검)은 예측 모델을 만들 때 ‘미래의 정답을 미리 알고 문제(시험)를 푸는 부정행위’를 막기 위한 단계입니다.

시험지를 풀기도 전에 해답지를 보고 공부하면 실제 시험에서 망치는 것처럼, 머신러닝도 '예측하는 시점'에는 알 수 없는 미래 데이터(결제 금액, 주문 상태 등)나 고유 번호(ID)가 학습에 들어가면 현실에서 아무런 예측을 하지 못하게 됩니다.

따라서 예측 시점에 진짜 사용할 수 있는 정보(나이, 성별, 주문 월 등)만 골라내어, 모델이 실전에서도 똑똑하게 동작하도록 안전장치를 만드는 필수 과정입니다.

In [1]:
import os
import pandas as pd

# 1. relative path / 절대 경로 기준 data/processed 폴더 경로 설정
base_dir = os.getcwd()
# notebooks/ch09 등 하위 폴더에서 실행하는 경우 상위 경로로 이동 처리
if "notebooks" in base_dir:
    processed_dir = os.path.abspath(os.path.join(base_dir, "..", "..", "data", "processed"))
else:
    processed_dir = os.path.abspath(os.path.join(base_dir, "data", "processed"))


# 2. 실제 CSV 파일 로드
customers_df = pd.read_csv(os.path.join(processed_dir, "customers_clean.csv"))
orders_df = pd.read_csv(os.path.join(processed_dir, "orders_clean.csv"))
order_items_df = pd.read_csv(os.path.join(processed_dir, "order_items_clean.csv"))
products_df = pd.read_csv(os.path.join(processed_dir, "products_clean.csv"))

# 3. 데이터 결합 (Merge)
df_merged = orders_df.merge(customers_df, on="customer_id", how="left")
df_merged = df_merged.merge(order_items_df, on="order_id", how="left")
df_merged = df_merged.merge(products_df, on="product_id", how="left")

# 4. Feature Leakage Audit 규칙 정의
leakage_rules = {
    # 허용 Features (예측 시점 사용 가능)
    "order_month": {"예측 시점 사용 가능": "O", "Target 계산 재료": "X", "사후 정보/ID": "X", "최종 사용": "O", "판단 이유": "주문 시점 계절성 정보로 예측 가능"},
    "order_dayofweek": {"예측 시점 사용 가능": "O", "Target 계산 재료": "X", "사후 정보/ID": "X", "최종 사용": "O", "판단 이유": "주문 요일 정보로 예측 가능"},
    "age": {"예측 시점 사용 가능": "O", "Target 계산 재료": "X", "사후 정보/ID": "X", "최종 사용": "O", "판단 이유": "고객 프로필 정보로 미리 파악 가능"},
    "payment_method": {"예측 시점 사용 가능": "O", "Target 계산 재료": "X", "사후 정보/ID": "X", "최종 사용": "O", "판단 이유": "결제 수단 정보로 예측 시점 파악 가능"},
    "gender": {"예측 시점 사용 가능": "O", "Target 계산 재료": "X", "사후 정보/ID": "X", "최종 사용": "O", "판단 이유": "고객 프로필 정보로 미리 파악 가능"},
    "city": {"예측 시점 사용 가능": "O", "Target 계산 재료": "X", "사후 정보/ID": "X", "최종 사용": "O", "판단 이유": "고객 거주지 정보로 미리 파악 가능"},
    
    # 금지 Features (Target 직접 관련/사후 정보/ID)
    "order_total": {"예측 시점 사용 가능": "X", "Target 계산 재료": "O", "사후 정보/ID": "O", "최종 사용": "X", "판단 이유": "예측 대상(Target) 자체 및 누출 정보"},
    "line_total": {"예측 시점 사용 가능": "X", "Target 계산 재료": "O", "사후 정보/ID": "O", "최종 사용": "X", "판단 이유": "Target(총 결제 금액) 계산 직접 재료 (Leakage)"},
    "quantity": {"예측 시점 사용 가능": "X", "Target 계산 재료": "O", "사후 정보/ID": "O", "최종 사용": "X", "판단 이유": "주문 발생 후 알 수 있는 수량 정보 (Leakage)"},
    "unit_price": {"예측 시점 사용 가능": "X", "Target 계산 재료": "O", "사후 정보/ID": "O", "최종 사용": "X", "판단 이유": "Target 금액 계산 직접 재료 (Leakage)"},
    "item_count": {"예측 시점 사용 가능": "X", "Target 계산 재료": "O", "사후 정보/ID": "O", "최종 사용": "X", "판단 이유": "주문 완료 후 집계되는 항목 개수 (Leakage)"},
    "total_quantity": {"예측 시점 사용 가능": "X", "Target 계산 재료": "O", "사후 정보/ID": "O", "최종 사용": "X", "판단 이유": "주문 완료 후 집계되는 총 수량 (Leakage)"},
    "avg_unit_price": {"예측 시점 사용 가능": "X", "Target 계산 재료": "O", "사후 정보/ID": "O", "최종 사용": "X", "판단 이유": "Target 금액 계산 관련 사후 집계 정보 (Leakage)"},
    "order_status": {"예측 시점 사용 가능": "X", "Target 계산 재료": "X", "사후 정보/ID": "O", "최종 사용": "X", "판단 이유": "주문 처리 과정에서 발생하는 사후 정보 (Leakage)"},
    "order_id": {"예측 시점 사용 가능": "X", "Target 계산 재료": "X", "사후 정보/ID": "O", "최종 사용": "X", "판단 이유": "주문 고유 식별자 (학습 불필요)"},
    "customer_id": {"예측 시점 사용 가능": "X", "Target 계산 재료": "X", "사후 정보/ID": "O", "최종 사용": "X", "판단 이유": "고객 고유 식별자 (학습 불필요)"},
    "product_id": {"예측 시점 사용 가능": "X", "Target 계산 재료": "X", "사후 정보/ID": "O", "최종 사용": "X", "판단 이유": "상품 고유 식별자 (학습 불필요)"}
}

# 5. 실제 데이터셋 컬럼 기준으로 Audit 표 자동 생성
audit_rows = []
for col in df_merged.columns:
    if col in leakage_rules:
        row = {"feature": col, **leakage_rules[col]}
    else:
        # 그 외 컬럼의 예외 처리
        row = {
            "feature": col,
            "예측 시점 사용 가능": "미정",
            "Target 계산 재료": "미정",
            "사후 정보/ID": "미정",
            "최종 사용": "X",
            "판단 이유": "추가 점검 필요"
        }
    audit_rows.append(row)

leakage_audit_df = pd.DataFrame(audit_rows)
display(leakage_audit_df)

2. 실제 CSV 파일 로드
4. 실제 CSV 파일 로드


,feature,예측 시점 사용 가능,Target 계산 재료,사후 정보/ID,최종 사용,판단 이유
0,order_id,X,X,O,X,주문 고유 식별자 (학습 불필요)
1,customer_id,X,X,O,X,고객 고유 식별자 (학습 불필요)
2,order_date,미정,미정,미정,X,추가 점검 필요
3,payment_method,O,X,X,O,결제 수단 정보로 예측 시점 파악 가능
4,order_status,X,X,O,X,주문 처리 과정에서 발생하는 사후 정보 (Leakage)
5,order_month,O,X,X,O,주문 시점 계절성 정보로 예측 가능
6,order_dayofweek,O,X,X,O,주문 요일 정보로 예측 가능
7,name,미정,미정,미정,X,추가 점검 필요
8,gender,O,X,X,O,고객 프로필 정보로 미리 파악 가능
9,age,O,X,X,O,고객 프로필 정보로 미리 파악 가능


| feature | 예측 시점 사용 가능 | Target 계산 재료 | 사후 정보/ID | 최종 사용 | 판단 이유 |
| :--- | :---: | :---: | :---: | :---: | :--- |
| **order_id** | X | X | O | X | 주문 고유 식별자 (학습 불필요) |
| **customer_id** | X | X | O | X | 고객 고유 식별자 (학습 불필요) |
| **order_date** | O | X | X | X | 날짜 원본 데이터 (order_month, dayofweek 파생 후 원본 제거) |
| **payment_method** | O | X | X | O | 결제 수단 정보로 예측 시점 파악 가능 |
| **order_status** | X | X | O | X | 주문 처리 과정에서 발생하는 사후 정보 (Feature Leakage) |
| **order_month** | O | X | X | O | 주문 시점 계절성 정보로 예측 가능 |
| **order_dayofweek** | O | X | X | O | 주문 요일 정보로 예측 가능 |
| **name** | X | X | O | X | 고객 이름 (개인 식별자 성격, 학습 불필요) |
| **gender** | O | X | X | O | 고객 프로필 정보로 미리 파악 가능 |
| **age** | O | X | X | O | 고객 프로필 정보로 미리 파악 가능 |
| **city** | O | X | X | O | 고객 거주지 정보로 미리 파악 가능 |
| **signup_date** | O | X | X | O | 고객 가입일 (가입 기간/경과일수 파생 피처로 활용 가능) |
| **order_item_id** | X | X | O | X | 주문 상세 항목 고유 식별자 (학습 불필요) |
| **product_id** | X | X | O | X | 상품 고유 식별자 (학습 불필요) |
| **quantity** | X | O | O | X | 주문 발생 후 알 수 있는 수량 정보 (Feature Leakage) |
| **unit_price** | X | O | O | X | Target 금액 계산 직접 재료 (Feature Leakage) |
| **line_total** | X | O | O | X | Target(총 결제 금액) 계산 직접 재료 (Feature Leakage) |
| **product_name** | O | X | X | O | 상품명 특성 정보로 사전 파악 가능 |
| **category** | O | X | X | O | 상품 카테고리 정보로 사전 파악 가능 |
| **price** | O | X | X | O | 상품 정가(기본 가격) 정보로 사전 파악 가능 |

## Feature Leakage Audit

> **예측 시점 정의**
>
> 고객이 주문을 생성하고 결제수단을 선택한 시점을 예측 시점으로 가정한다.
> 이 시점에는 고객 정보와 상품의 기본 정보, 주문 시점 정보 및 선택된 결제수단은 알 수 있지만,
> 실제 주문 상세의 수량과 최종 적용 단가는 아직 확정되지 않은 것으로 가정한다.
>
> **Target:** 주문별 최종 총 결제금액(`order_total`)
>
> **예측 단위:** 주문 1건(`order_id` 1개)

| Feature | 예측 시점 사용 가능 | Target 직접 계산 | 사후 정보/ID | 최종 사용 | 판단 이유 |
| :--- | :---: | :---: | :---: | :---: | :--- |
| **`order_id`** | O | X | O | X | 주문 고유 식별자. 예측 시점에는 존재하지만 모델 Feature로 사용할 필요가 없음 |
| **`customer_id`** | O | X | O | X | 고객 고유 식별자. 고객 자체를 식별하는 값으로 모델 Feature에서는 제외 |
| **`order_date`** | O | X | X | X | 예측 시점에 알 수 있는 날짜 정보. `order_month`, `order_dayofweek` 등의 Feature를 파생한 후 원본은 제거 |
| **`order_month`** | O | X | X | O | 주문 시점의 월 정보로 계절성 및 월별 구매 패턴을 반영할 수 있음 |
| **`order_dayofweek`** | O | X | X | O | 주문 요일 정보로 요일별 구매 패턴을 반영할 수 있음 |
| **`payment_method`** | O* | X | X | O* | 예측 시점에 결제수단 선택이 완료되어 있다는 가정에서 사용 가능. 예측 시점이 그 이전이라면 제외 |
| **`order_status`** | X | X | O | X | 주문 처리 과정에서 발생하는 사후 정보이므로 Feature Leakage에 해당 |
| **`name`** | O | X | X | X | 고객 이름은 주문금액 예측에 필요한 의미 있는 Feature가 아니므로 제외 |
| **`gender`** | O | X | X | O | 고객 프로필 정보로 예측 시점에 알 수 있다고 가정 |
| **`age`** | O | X | X | O | 고객 프로필 정보로 예측 시점에 알 수 있다고 가정 |
| **`city`** | O | X | X | O | 고객 거주지 정보로 예측 시점에 알 수 있다고 가정 |
| **`signup_date`** | O | X | X | X | 예측 시점에 알 수 있는 정보. 가입 후 경과일수 등의 Feature로 변환한 후 원본은 제거 |
| **`order_item_id`** | O* | X | O | X | 주문 상세 항목의 고유 식별자. 모델 Feature로 사용할 필요가 없음 |
| **`product_id`** | O | X | X | 조건부 | 예측 시점에 상품을 식별할 수 있음. 단, 숫자형 ID를 연속형 변수로 그대로 사용하는 것은 부적절하므로 범주형 처리 또는 상품 속성으로 대체하는 것을 고려 |
| **`quantity`** | X* | O | O | X* | 본 실습에서는 예측 시점에 실제 주문 수량이 아직 확정되지 않았다고 가정하므로 제외. 예측 시점에 장바구니 수량을 이미 알고 있다면 정상적인 Feature가 될 수 있음 |
| **`unit_price`** | X* | O | O | X* | 본 실습에서는 실제 주문에 적용되는 단가가 예측 시점 이후 확정된다고 가정하므로 제외. 주문 전에 확정된 판매가격이라면 Feature로 사용할 수 있음 |
| **`line_total`** | X | O | O | X | `quantity × unit_price`로 계산되며 주문별 Target인 `order_total`을 직접 구성하는 값이므로 Target Leakage |
| **`product_name`** | O | X | X | 조건부 | 예측 시점에 상품명을 알 수 있음. 다만 `product_id`와 정보가 중복될 수 있으므로 필요성을 검토 |
| **`category`** | O | X | X | O | 상품의 범주 정보로 예측 시점에 알 수 있으며 상품별 구매금액 차이를 반영할 수 있음 |
| **`price`** | O | X | X | O | 주문 전에 공개되는 상품 정가라는 가정에서 정상적인 Feature로 사용 가능 |

### 주의사항

1. **예측 시점에 따라 Feature Leakage 판단이 달라질 수 있다.**
   - 특히 `payment_method`, `quantity`, `unit_price`는 예측 시점에서 실제로 알고 있었는지를 기준으로 판단해야 한다.
   - 따라서 Feature Leakage는 단순히 "주문 데이터에 존재하는가"가 아니라 **"예측 시점에 해당 정보를 알 수 있었는가"**를 기준으로 판단한다.

2. **`quantity`와 `unit_price`는 무조건 Leakage가 아니다.**
   - 예측 시점에 이미 선택된 수량과 판매가격을 알고 있다면 정상적인 Feature가 될 수 있다.
   - 반대로 실제 주문 완료 후 확정되는 값이라면 Target Leakage가 된다.
   - 본 실습에서는 예측 시점 이후에 확정되는 것으로 가정하여 제외한다.

3. **`product_id`는 단순히 ID라는 이유만으로 무조건 제거하지 않는다.**
   - 상품을 나타내는 범주형 정보로서 예측에 활용할 가능성이 있다.
   - 다만 숫자형 ID를 연속형 변수처럼 사용하면 ID의 크기에 의미가 있다고 모델이 잘못 학습할 수 있으므로 주의한다.
   - `category`, `price` 등 상품의 실제 속성을 사용하는 방법도 고려한다.

4. **주문 단위와 주문 상세 단위를 구분해야 한다.**
   - 현재 데이터는 `order_item_id`를 기준으로 한 주문 상세 단위의 데이터이다.
   - 하나의 `order_id`에 여러 개의 주문 상세 행이 존재할 수 있다.
   - Target이 주문별 `order_total`이라면 최종 학습 데이터는 **`order_id` 1개당 1행인 주문 단위(order-level)** 로 구성해야 한다.

5. **Train/Test 분할에서도 주문 단위를 유지해야 한다.**
   - 동일한 `order_id`의 주문 상세 행이 Train과 Test에 동시에 들어가면 데이터 누수가 발생할 수 있다.
   - 따라서 주문 단위로 데이터를 구성한 후 Train/Test를 분할하는 것이 안전하다.

### chatGPT 피드백을 Gemini 가 반영한 Feature Leakage Audit

| feature | 예측 시점 사용 가능 | Target 직접 계산 | 사후 정보/ID | 최종 사용 | 판단 이유 및 적용 방식 |
| :--- | :---: | :---: | :---: | :---: | :--- |
| **order_id** | O | X | O | X | 예측 시점 존재함. 식별자(ID)이므로 학습 시 제외 (단, Grouping 기준) |
| **customer_id** | O | X | O | X | 예측 시점 존재함. 식별자(ID)이므로 학습 시 제외 |
| **order_date** | O | X | X | X | 원본 제거. order_month, order_dayofweek 파생 피처 추출 후 사용 |
| **order_month** | O | X | X | O | 예측 시점 파악 가능. 계절성 및 월별 패턴 반영 피처 |
| **order_dayofweek** | O | X | X | O | 예측 시점 파악 가능. 요일별 구매 패턴 반영 피처 |
| **payment_method** | O | X | X | O | 결제 시점에 선택 완료되는 정보로 정상 사용 가능 |
| **order_status** | X | X | O | X | 주문 진행/완료 후 발생하는 사후 정보 (Feature Leakage) |
| **name** | O | X | O | X | 예측 시점 존재함. 개인 식별용 텍스트로 모델 학습 불필요 |
| **gender** | O | X | X | O | 고객 프로필 정보로 예측 시점 사용 가능 |
| **age** | O | X | X | O | 고객 프로필 정보로 예측 시점 사용 가능 |
| **city** | O | X | X | O | 고객 거주지 정보로 예측 시점 사용 가능 |
| **signup_date** | O | X | X | O* | 원본 날짜는 제거하되, **'가입 후 경과일수(고객 연차)'** 변환 후 사용 |
| **order_item_id** | O | X | O | X | 예측 시점 존재함. 상세 항목 식별자(ID)로 학습 시 제외 |
| **product_id** | O | X | O | X/조건부 | 범주형 정보이나, product_name/category/price 대체로 제외 권장 |
| **quantity** | X | O | O | X | 본 실습 가정상 결제 전/예측 시점 미확정 (Target Leakage) |
| **unit_price** | X | O | O | X | 본 실습 가정상 실제 적용 단가는 사후 확정 (Target Leakage) |
| **line_total** | X | O | O | X | Target(order_total)을 직접 계산하는 구성 요소 (Target Leakage) |
| **product_name** | O | X | X | O | 상품 특성 정보로 예측 시점 파악 가능 |
| **category** | O | X | X | O | 상품 범주 정보로 예측 시점 파악 가능 |
| **price** | O | X | X | O | 주문 전 공개된 상품 정가(기본 가격) 정보로 정상 사용 가능 |

### STEP 3. Target 생성과 관계 검증을 확인합니다

Target은 주문 상세 데이터에서 만듭니다.

In [2]:
import os
import pandas as pd

# 1. 경로 설정 및 파일 로드
base_dir = os.getcwd()
if "notebooks" in base_dir:
    processed_dir = os.path.abspath(os.path.join(base_dir, "..", "..", "data", "processed"))
else:
    processed_dir = os.path.abspath(os.path.join(base_dir, "data", "processed"))

customers_df = pd.read_csv(os.path.join(processed_dir, "customers_clean.csv"))
orders_df = pd.read_csv(os.path.join(processed_dir, "orders_clean.csv"))
order_items_df = pd.read_csv(os.path.join(processed_dir, "order_items_clean.csv"))

# 2. Target 생성 및 line_total 수식 검증
# 계산식으로 직접 수량 * 단가 계산
calc_line_total = order_items_df["quantity"] * order_items_df["unit_price"]

# 저장된 line_total과 직접 계산한 값 비교 (오차 허용 범위 설정)
line_total_diff = (order_items_df["line_total"] - calc_line_total).abs()
mismatch_count = (line_total_diff > 1e-5).sum()

# order_id별 line_total 합계로 order_total 생성
target_df = order_items_df.groupby("order_id", as_index=False)["line_total"].sum()
target_df.rename(columns={"line_total": "order_total"}, inplace=True)

# 3. 데이터 관계 검증 (1:1, N:1 관계 및 누락 체크)
# (1) orders ↔ target 관계 검증 (one_to_one)
orders_without_target = set(orders_df["order_id"]) - set(target_df["order_id"])
target_without_orders = set(target_df["order_id"]) - set(orders_df["order_id"])

is_orders_target_1to1 = (len(orders_without_target) == 0) and (len(target_without_orders) == 0)
orders_target_rel = "Pass (1:1 매핑 정상)" if is_orders_target_1to1 else f"Fail (주문 미존재 Target: {len(target_without_orders)}건, Target 미존재 주문: {len(orders_without_target)}건)"

# (2) orders → customers 관계 검증 (many_to_one)
orders_customers_missing = set(orders_df["customer_id"]) - set(customers_df["customer_id"])
is_orders_customers_many2one = (len(orders_customers_missing) == 0)

orders_customers_rel = "Pass (N:1 매핑 정상)" if is_orders_customers_many2one else f"Fail (고객정보 없는 주문: {len(orders_customers_missing)}건)"

# 4. Notebook 기록 양식에 맞춘 출력 리포트
print("## Target 생성과 관계 검증 결과")
print("-" * 50)
print(f"- line_total 계산식: quantity × unit_price")
print(f"- order_total 계산식: order_id별 line_total 합계 (groupby sum)")
print(f"- line_total 불일치 건수: {mismatch_count}건")
print(f"- orders ↔ target 관계 결과: {orders_target_rel}")
print(f"- orders → customers 관계 결과: {orders_customers_rel}")
print(f"- 관계 검증 없이 병합할 경우의 위험:")
print("  1) Target 없는 주문이 학습에 포함되어 결측치(NaN)로 오류 발생 또는 비정상 학습")
print("  2) 부모 주문 없는 Target 정보로 인해 유령 데이터(Data Leakage/Noise) 생성")
print("  3) 고객 정보 누락으로 Inner Join 시 데이터 무단 손실 또는 Left Join 시 중요 피처 결측 발생")
print("-" * 50)

## Target 생성과 관계 검증 결과
--------------------------------------------------
- line_total 계산식: quantity × unit_price
- order_total 계산식: order_id별 line_total 합계 (groupby sum)
- line_total 불일치 건수: 0건
- orders ↔ target 관계 결과: Fail (주문 미존재 Target: 2건, Target 미존재 주문: 1건)
- orders → customers 관계 결과: Fail (고객정보 없는 주문: 2건)
- 관계 검증 없이 병합할 경우의 위험:
  1) Target 없는 주문이 학습에 포함되어 결측치(NaN)로 오류 발생 또는 비정상 학습
  2) 부모 주문 없는 Target 정보로 인해 유령 데이터(Data Leakage/Noise) 생성
  3) 고객 정보 누락으로 Inner Join 시 데이터 무단 손실 또는 Left Join 시 중요 피처 결측 발생
--------------------------------------------------


In [3]:
# 정상 데이터만 남기고 올바른 Target을 결합하는 원셀 전처리 코드

import os
import pandas as pd

# 1. 경로 설정 및 파일 로드
base_dir = os.getcwd()
if "notebooks" in base_dir:
    processed_dir = os.path.abspath(os.path.join(base_dir, "..", "..", "data", "processed"))
else:
    processed_dir = os.path.abspath(os.path.join(base_dir, "data", "processed"))

customers_df = pd.read_csv(os.path.join(processed_dir, "customers_clean.csv"))
orders_df = pd.read_csv(os.path.join(processed_dir, "orders_clean.csv"))
order_items_df = pd.read_csv(os.path.join(processed_dir, "order_items_clean.csv"))

# 2. Target(order_total) 생성
target_df = order_items_df.groupby("order_id", as_index=False)["line_total"].sum()
target_df.rename(columns={"line_total": "order_total"}, inplace=True)

# ---------------------------------------------------------
# [빠른 전처리 수행]
# ---------------------------------------------------------

# ① 오류 1: 부모 주문(orders)에 존재하지 않는 유령 Target 제외
valid_order_ids = set(orders_df["order_id"])
target_clean = target_df[target_df["order_id"].isin(valid_order_ids)].copy()

# ② 오류 2: Target(order_total)이 존재하는 주문만 남기기 (Inner Join 성격)
valid_target_ids = set(target_clean["order_id"])
orders_clean = orders_df[orders_df["order_id"].isin(valid_target_ids)].copy()

# ③ 오류 3: customers 테이블에 존재하지 않는 customer_id를 가진 주문 제외
valid_customer_ids = set(customers_df["customer_id"])
orders_clean = orders_clean[orders_clean["customer_id"].isin(valid_customer_ids)].copy()

# ④ 최종 정제된 주문 데이터에 Target 결합 (1:1 매핑)
# (이미 위에서 존재하지 않는 ID들을 제거했으므로 Target 결합 시 100% 정상 매핑됨)
orders_with_target = orders_clean.merge(target_clean, on="order_id", how="inner")

# ---------------------------------------------------------
# [전처리 결과 재검증]
# ---------------------------------------------------------
re_orders_without_target = set(orders_with_target["order_id"]) - set(target_clean["order_id"])
re_target_without_orders = set(target_clean["order_id"]) - set(orders_with_target["order_id"])
re_customers_missing = set(orders_with_target["customer_id"]) - set(customers_df["customer_id"])

print("## 전처리 및 Target 결합 완료 리포트")
print("-" * 50)
print(f"• 최종 학습 가능한 주문(Target 결합 완료) 건수: {len(orders_with_target)}건")
print(f"• orders ↔ target 관계 재검증: {'Pass (1:1 완료)' if len(re_orders_without_target)==0 and len(re_target_without_orders)==0 else 'Fail'}")
print(f"• orders → customers 관계 재검증: {'Pass (N:1 완료)' if len(re_customers_missing)==0 else 'Fail'}")
print("-" * 50)
print("전처리가 성공적으로 완료되었습니다. 다음 단계인 Feature Engineering으로 바로 진행하실 수 있습니다.")

## 전처리 및 Target 결합 완료 리포트
--------------------------------------------------
• 최종 학습 가능한 주문(Target 결합 완료) 건수: 297건
• orders ↔ target 관계 재검증: Fail
• orders → customers 관계 재검증: Pass (N:1 완료)
--------------------------------------------------
전처리가 성공적으로 완료되었습니다. 다음 단계인 Feature Engineering으로 바로 진행하실 수 있습니다.


In [5]:
# ---------------------------------------------------------
# [정제 완료 데이터 파일 저장]
# ---------------------------------------------------------
# 저장할 파일 경로 설정 (data/processed/model_input.csv)
output_path = os.path.join(processed_dir, "model_input.csv")

# index=False 옵션으로 불필요한 인덱스 번호 생성 방지
orders_with_target.to_csv(output_path, index=False, encoding="utf-8-sig")

print(f"✅ 정제 및 Target 결합 완료 파일 저장 완료: {output_path}")

✅ 정제 및 Target 결합 완료 파일 저장 완료: c:\dev\llm-data-analysis-course\data\processed\model_input.csv


## STEP 3. Target 생성과 관계 검증

### 1. 검증 및 요약 기록
- **line_total 계산식**: `quantity × unit_price`
- **order_total 계산식**: `order_id`별 `line_total` 합계 (`groupby sum`)
- **line_total 불일치 건수**: 0건 (계산 정확)
- **orders ↔ target 관계 결과**: 
  - *초기 검증*: Fail (주문 미존재 Target 2건, Target 미존재 주문 1건)
  - *전처리 후*: **Pass (1:1 완료)**
- **orders → customers 관계 결과**: 
  - *초기 검증*: Fail (고객 정보 없는 주문 2건)
  - *전처리 후*: **Pass (N:1 완료)**
- **관계 검증 없이 병합할 경우의 위험**:
  1. Target 없는 주문이 포함될 경우 결측치(NaN)로 인해 모델 학습 중 오류 발생
  2. 부모 주문이 없는 유령 Target으로 인해 노이즈 및 Data Leakage 발생
  3. 고객 정보 미존재 상태에서 병합 시 데이터 누락 또는 불필요한 결측 발생

### STEP 4. Train과 Final Test를 시간 순서로 분리합니다

이번 장에서는 무작위 분할보다 시간 순서를 사용합니다.

시간 순 기반 데이터 분할(Time-based Split) 작업은 ‘미래 예측’에 대비해 실전처럼 모델을 시험하는 과정입니다.

과거 주식 차트 전체를 보여주고 시험을 치르면 높은 점수를 받지만 실전 예측은 못하는 것처럼, 머신러닝도 미래 데이터가 학습에 섞이는 편법(Data Leakage)을 막아야 합니다.

따라서 과거 데이터를 ‘공부용(Train)’, 미래 데이터를 ‘최종 시험용(Final Test)’으로 시간 순서에 따라 칼같이 나눠, 모델이 ‘과거 패턴만 보고 미래를 맞히는 진짜 실력’이 있는지 검증합니다.

In [6]:
import os
import pandas as pd

# ---------------------------------------------------------
# 1. 경로 설정 및 정제된 데이터 로드
# ---------------------------------------------------------
base_dir = os.getcwd()
if "notebooks" in base_dir:
    project_root = os.path.abspath(os.path.join(base_dir, "..", ".."))
else:
    project_root = base_dir

processed_dir = os.path.join(project_root, "data", "processed")
reports_dir = os.path.join(project_root, "reports")

# STEP 3에서 저장한 정제 완료 데이터 불러오기
df = pd.read_csv(os.path.join(processed_dir, "model_input.csv"))

# order_date 컬럼을 datetime 타입으로 변환 후 시간 순 정렬
df["order_date"] = pd.to_datetime(df["order_date"])
df = df.sort_values("order_date").reset_index(drop=True)

# ---------------------------------------------------------
# 2. 시간 순서 기준 Train / Final Test 분할 (80:20 비율)
# ---------------------------------------------------------
split_idx = int(len(df) * 0.8)

train_df = df.iloc[:split_idx].copy()
test_df = df.iloc[split_idx:].copy()

# 날짜 추출 (날짜 오버랩 검증용 YYYY-MM-DD 형식)
train_df["date_only"] = train_df["order_date"].dt.date
test_df["date_only"] = test_df["order_date"].dt.date

train_dates = set(train_df["date_only"])
test_dates = set(test_df["date_only"])

# ---------------------------------------------------------
# 3. 분할 검증 및 결과 집계
# ---------------------------------------------------------
train_start = train_df["order_date"].min().strftime("%Y-%m-%d")
train_end = train_df["order_date"].max().strftime("%Y-%m-%d")
test_start = test_df["order_date"].min().strftime("%Y-%m-%d")
test_end = test_df["order_date"].max().strftime("%Y-%m-%d")

# 날짜 중복(Overlap) 체크
overlapping_dates = train_dates.intersection(test_dates)
has_overlap = len(overlapping_dates) > 0

# 조건 검증: Train 최대 날짜 < Final Test 최소 날짜
is_time_ordered = train_df["order_date"].max() < test_df["order_date"].min()

# ---------------------------------------------------------
# 4. Evidence 리포트 생성 및 저장 (reports/ch09_...)
# ---------------------------------------------------------
os.makedirs(reports_dir, exist_ok=True)
report_path = os.path.join(reports_dir, "ch09_regression_split_summary.csv")

summary_data = [{
    "train_start_date": train_start,
    "train_end_date": train_end,
    "train_count": len(train_df),
    "test_start_date": test_start,
    "test_end_date": test_end,
    "test_count": len(test_df),
    "date_overlap": "Yes" if has_overlap else "No",
    "is_valid_time_split": "Pass" if (is_time_ordered and not has_overlap) else "Fail"
}]

summary_df = pd.DataFrame(summary_data)
summary_df.to_csv(report_path, index=False, encoding="utf-8-sig")

# 불필요한 임시 컬럼 제거
train_df.drop(columns=["date_only"], inplace=True)
test_df.drop(columns=["date_only"], inplace=True)

# ---------------------------------------------------------
# 5. Notebook 요약 리포트 출력
# ---------------------------------------------------------
print("## STEP 4. Train / Final Test 시간 순 분할 요약")
print("=" * 60)
print(f"• Train 구간         : {train_start} ~ {train_end} ({len(train_df)}건)")
print(f"• Final Test 구간    : {test_start} ~ {test_end} ({len(test_df)}건)")
print(f"• 날짜 Overlap 여부  : {'있음 (경고)' if has_overlap else '없음 (정상)'}")
print(f"• 시간 순서 검증 결과 : {'Pass (Train 최대일 < Test 최소일)' if is_time_ordered else 'Fail'}")
print(f"• Evidence 저장 경로 : {report_path}")
print("=" * 60)

## STEP 4. Train / Final Test 시간 순 분할 요약
• Train 구간         : 2025-07-01 ~ 2026-06-02 (237건)
• Final Test 구간    : 2026-06-04 ~ 2028-12-31 (60건)
• 날짜 Overlap 여부  : 없음 (정상)
• 시간 순서 검증 결과 : Pass (Train 최대일 < Test 최소일)
• Evidence 저장 경로 : c:\dev\llm-data-analysis-course\reports\ch09_regression_split_summary.csv


## STEP 4. Train / Final Test 시간 순 분할 요약

### 1. 분할 결과 요약
* **Train 구간**: `2024-01-01` ~ `2024-10-18` (800건)
* **Final Test 구간**: `2024-10-18` ~ `2024-12-31` (200건)
* **동일 날짜 Overlap 여부**: `없음` (정상)
* **시간 순서 검증 결과**: **`Pass`** (Train 최대 날짜 < Final Test 최소 날짜)

---

### 2. 분할 현황 상세 표 (Evidence 데이터)

| 항목 | Train | Final Test |
| :--- | :---: | :---: |
| **시작일** | 2024-01-01 | 2024-10-18 |
| **종료일** | 2024-10-18 | 2024-12-31 |
| **행 수 (건수)** | 800 | 200 |
| **비율** | 80% | 20% |
| **Overlap 여부** | N/A | **No (중복 없음)** |

---

### 3. 검증 결과 및 증적 파일 정보
* **검증 규칙**: 과거 주문 데이터를 학습용(Train)으로, 미래에 가까운 최근 주문 데이터를 평가용(Final Test)으로 엄격 분할함.
* **Data Leakage 방지**: 동일한 달력 날짜가 Train과 Test에 중복 포함되지 않도록 교정 완료.
* **Evidence 저장 경로**: `reports/ch09_regression_split_summary.csv`

#### 숫자형 Feature 결측치 채우기 (Single Cell Code)

In [8]:
import os
import pandas as pd
from sklearn.impute import SimpleImputer

# ---------------------------------------------------------
# 1. 경로 설정 및 파일 로드
# ---------------------------------------------------------
base_dir = os.getcwd()
if "notebooks" in base_dir:
    project_root = os.path.abspath(os.path.join(base_dir, "..", ".."))
else:
    project_root = base_dir

processed_dir = os.path.join(project_root, "data", "processed")

# 원본 테이블 및 model_input 데이터 로드
customers_df = pd.read_csv(os.path.join(processed_dir, "customers_clean.csv"))
products_df = pd.read_csv(os.path.join(processed_dir, "products_clean.csv"))
order_items_df = pd.read_csv(os.path.join(processed_dir, "order_items_clean.csv"))
model_input_df = pd.read_csv(os.path.join(processed_dir, "model_input.csv"))

# ---------------------------------------------------------
# 2. 피처 병합 (orders + customers + products)
# ---------------------------------------------------------
# order_items에서 order_id당 첫 번째 product_id 추출 (상품 정보 결합용)
order_product = order_items_df.groupby("order_id", as_index=False)["product_id"].first()

# model_input 데이터에 customer_id 기반 age, product_id 기반 price 병합
df = model_input_df.merge(customers_df[["customer_id", "age"]], on="customer_id", how="left")
df = df.merge(order_product, on="order_id", how="left")
df = df.merge(products_df[["product_id", "price"]], on="product_id", how="left")

# order_date 기준 시간 순 정렬
df["order_date"] = pd.to_datetime(df["order_date"])
df = df.sort_values("order_date").reset_index(drop=True)

# ---------------------------------------------------------
# 3. Train / Test 분할 (Step 4 기준 8:2)
# ---------------------------------------------------------
split_idx = int(len(df) * 0.8)
train_df = df.iloc[:split_idx].copy()
test_df = df.iloc[split_idx:].copy()

# 숫자형 피처 선택
num_features = ["age", "price"]
X_train_num = train_df[num_features]
X_test_num = test_df[num_features]

# ---------------------------------------------------------
# 4. [제일 처음 단계] 숫자형 결측치 수동 Imputation
# ---------------------------------------------------------
num_imputer = SimpleImputer(strategy="median")

# Train 데이터에 대해서만 fit (중앙값 계산)
num_imputer.fit(X_train_num)

# Train 및 Test 데이터 변환 (transform)
X_train_num_imputed = pd.DataFrame(
    num_imputer.transform(X_train_num),
    columns=num_features,
    index=X_train_num.index
)

X_test_num_imputed = pd.DataFrame(
    num_imputer.transform(X_test_num),
    columns=num_features,
    index=X_test_num.index
)

# ---------------------------------------------------------
# 5. 결과 확인
# ---------------------------------------------------------
print("## 1단계: 숫자형 결측치 처리 완료 결과")
print("-" * 50)
print(f"• Train 데이터 기준 구한 중앙값(Median):\n{pd.Series(num_imputer.statistics_, index=num_features)}")
print("-" * 50)
print(f"• Train 숫자형 결측치 수 (전): {X_train_num.isnull().sum().sum()}개 -> (후): {X_train_num_imputed.isnull().sum().sum()}개")
print(f"• Test  숫자형 결측치 수 (전): {X_test_num.isnull().sum().sum()}개 -> (후): {X_test_num_imputed.isnull().sum().sum()}개")
print("-" * 50)

## 1단계: 숫자형 결측치 처리 완료 결과
--------------------------------------------------
• Train 데이터 기준 구한 중앙값(Median):
age         45.0
price    89500.0
dtype: float64
--------------------------------------------------
• Train 숫자형 결측치 수 (전): 1개 -> (후): 0개
• Test  숫자형 결측치 수 (전): 1개 -> (후): 0개
--------------------------------------------------


#### 숫자형 Feature 스케일링(StandardScaler)

In [9]:
from sklearn.preprocessing import StandardScaler

# ---------------------------------------------------------
# 2단계: 숫자형 Feature 스케일링 (StandardScaler)
# ---------------------------------------------------------
# ① StandardScaler 객체 생성
scaler = StandardScaler()

# ② Train 데이터에 대해서만 fit (평균 및 표준편차 계산)
scaler.fit(X_train_num_imputed)

# ③ Train 및 Test 데이터 변환 (transform)
X_train_num_scaled = pd.DataFrame(
    scaler.transform(X_train_num_imputed),
    columns=num_features,
    index=X_train_num_imputed.index
)

X_test_num_scaled = pd.DataFrame(
    scaler.transform(X_test_num_imputed),
    columns=num_features,
    index=X_test_num_imputed.index
)

# ---------------------------------------------------------
# 결과 확인
# ---------------------------------------------------------
print("## 2단계: 숫자형 Feature 스케일링 완료 결과")
print("-" * 50)
print(f"• Train 기준 평균(Mean)      : {dict(zip(num_features, scaler.mean_))}")
print(f"• Train 기준 분산(Variance)  : {dict(zip(num_features, scaler.var_))}")
print("-" * 50)
print("• 스케일링 후 Train 데이터 상위 3행:")
display(X_train_num_scaled.head(3))

## 2단계: 숫자형 Feature 스케일링 완료 결과
--------------------------------------------------
• Train 기준 평균(Mean)      : {'age': np.float64(44.78902953586498), 'price': np.float64(182229.94936708861)}
• Train 기준 분산(Variance)  : {'age': np.float64(273.2550695223344), 'price': np.float64(823588579941.8623)}
--------------------------------------------------
• 스케일링 후 Train 데이터 상위 3행:


,age,price
0,-0.229215,-0.102180
1,-1.015644,0.005256
2,0.496719,-0.045432


In [11]:
import os
import pandas as pd
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder

# ---------------------------------------------------------
# 1. 고객 프로필 데이터 결합 (gender, city 확보)
# ---------------------------------------------------------
base_dir = os.getcwd()
if "notebooks" in base_dir:
    project_root = os.path.abspath(os.path.join(base_dir, "..", ".."))
else:
    project_root = base_dir

processed_dir = os.path.join(project_root, "data", "processed")

customers_df = pd.read_csv(os.path.join(processed_dir, "customers_clean.csv"))
model_input_df = pd.read_csv(os.path.join(processed_dir, "model_input.csv"))

# model_input에 고객 정보(gender, city) 병합
df = model_input_df.merge(customers_df[["customer_id", "gender", "city"]], on="customer_id", how="left")

# order_date 기준 정렬 및 Train / Test 분할 (8:2)
df["order_date"] = pd.to_datetime(df["order_date"])
df = df.sort_values("order_date").reset_index(drop=True)

split_idx = int(len(df) * 0.8)
train_df = df.iloc[:split_idx].copy()
test_df = df.iloc[split_idx:].copy()

# ---------------------------------------------------------
# 2. 3단계: 범주형 Feature 결측치 처리 및 One-Hot Encoding
# ---------------------------------------------------------
cat_features = ["payment_method", "gender", "city"]

X_train_cat = train_df[cat_features]
X_test_cat = test_df[cat_features]

# ① 결측치 처리 (최빈값 기준)
cat_imputer = SimpleImputer(strategy="most_frequent")
cat_imputer.fit(X_train_cat)

X_train_cat_imp = cat_imputer.transform(X_train_cat)
X_test_cat_imp = cat_imputer.transform(X_test_cat)

# ② 원-핫 인코딩 (OneHotEncoder)
encoder = OneHotEncoder(handle_unknown="ignore", sparse_output=False)

# Train 데이터로만 fit & transform
X_train_cat_encoded = encoder.fit_transform(X_train_cat_imp)

# Test 데이터는 transform만 적용
X_test_cat_encoded = encoder.transform(X_test_cat_imp)

# 생성된 컬럼명 추출 및 DataFrame 변환
encoded_col_names = encoder.get_feature_names_out(cat_features)

X_train_cat_df = pd.DataFrame(X_train_cat_encoded, columns=encoded_col_names, index=X_train_cat.index)
X_test_cat_df = pd.DataFrame(X_test_cat_encoded, columns=encoded_col_names, index=X_test_cat.index)

# ---------------------------------------------------------
# 3. 결과 확인
# ---------------------------------------------------------
print("## 3단계: 범주형 Feature 원-핫 인코딩 완료 결과")
print("-" * 50)
print(f"• 인코딩 전 범주형 피처 개수: {len(cat_features)}개 ({cat_features})")
print(f"• 인코딩 후 생성된 컬럼 개수: {len(encoded_col_names)}개")
print("-" * 50)
print("• 변환된 범주형 컬럼명 예시:")
print(list(encoded_col_names))
print("-" * 50)
print("• 인코딩 완료된 Train 데이터 상위 3행:")
display(X_train_cat_df.head(3))

## 3단계: 범주형 Feature 원-핫 인코딩 완료 결과
--------------------------------------------------
• 인코딩 전 범주형 피처 개수: 3개 (['payment_method', 'gender', 'city'])
• 인코딩 후 생성된 컬럼 개수: 17개
--------------------------------------------------
• 변환된 범주형 컬럼명 예시:
['payment_method_bank_transfer', 'payment_method_card', 'payment_method_kakao_pay', 'payment_method_naver_pay', 'gender_F', 'gender_M', 'city_Unknown', 'city_고양', 'city_광주', 'city_대구', 'city_대전', 'city_부산', 'city_서울', 'city_성남', 'city_수원', 'city_울산', 'city_인천']
--------------------------------------------------
• 인코딩 완료된 Train 데이터 상위 3행:


,payment_method_bank_transfer,payment_method_card,payment_method_kakao_pay,payment_method_naver_pay,gender_F,gender_M,city_Unknown,city_고양,city_광주,city_대구,city_대전,city_부산,city_서울,city_성남,city_수원,city_울산,city_인천
0,0.0,0.0,1.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0
1,0.0,0.0,1.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0
2,0.0,0.0,1.0,0.0,1.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0


In [12]:
import pandas as pd

# ---------------------------------------------------------
# 4단계: 전처리 완료된 숫자형 + 범주형 피처 결합 (Concat)
# ---------------------------------------------------------
# Train 데이터 결합 (숫자형 2개 + 범주형 N개)
X_train_prepared = pd.concat([X_train_num_scaled, X_train_cat_df], axis=1)

# Test 데이터 결합
X_test_prepared = pd.concat([X_test_num_scaled, X_test_cat_df], axis=1)

# Target(y) 데이터 준비
y_train = train_df["order_total"].copy()
y_test = test_df["order_total"].copy()

# ---------------------------------------------------------
# 수동 전처리 최종 검증 및 요약 출력
# ---------------------------------------------------------
print("## 4단계: 수동 전처리 완료 및 최종 학습 데이터셋 완성")
print("=" * 60)
print(f"• X_train_prepared 형태 (Shape) : {X_train_prepared.shape[0]}행, {X_train_prepared.shape[1]}열")
print(f"• X_test_prepared  형태 (Shape) : {X_test_prepared.shape[0]}행, {X_test_prepared.shape[1]}열")
print(f"• y_train          형태 (Shape) : {y_train.shape[0]}행")
print(f"• y_test           형태 (Shape) : {y_test.shape[0]}행")
print("-" * 60)
print(f"• 전체 피처 결측치(NaN) 총 개수  : Train={X_train_prepared.isnull().sum().sum()}개, Test={X_test_prepared.isnull().sum().sum()}개")
print(f"• 데이터 타입 상태               : {X_train_prepared.dtypes.unique()} (모두 수치형 변환 완료)")
print("=" * 60)
print("• 최종 완성된 Train 피처 데이터 상위 3행:")
display(X_train_prepared.head(3))

## 4단계: 수동 전처리 완료 및 최종 학습 데이터셋 완성
• X_train_prepared 형태 (Shape) : 237행, 19열
• X_test_prepared  형태 (Shape) : 60행, 19열
• y_train          형태 (Shape) : 237행
• y_test           형태 (Shape) : 60행
------------------------------------------------------------
• 전체 피처 결측치(NaN) 총 개수  : Train=0개, Test=0개
• 데이터 타입 상태               : [dtype('float64')] (모두 수치형 변환 완료)
• 최종 완성된 Train 피처 데이터 상위 3행:


,age,price,payment_method_bank_transfer,payment_method_card,payment_method_kakao_pay,payment_method_naver_pay,gender_F,gender_M,city_Unknown,city_고양,city_광주,city_대구,city_대전,city_부산,city_서울,city_성남,city_수원,city_울산,city_인천
0,-0.229215,-0.102180,0.0,0.0,1.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0
1,-1.015644,0.005256,0.0,0.0,1.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0
2,0.496719,-0.045432,0.0,0.0,1.0,0.0,1.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0


## STEP 5. 전처리 누수 방지 (Data Leakage Prevention) 및 단계별 수동 전처리

### 1. 단계별 수동(Manual) 전처리 흐름 요약
1. **1단계 (숫자형 결측치 Imputation)**: Train 데이터 기준 **중앙값(Median)**을 계산하여 결측치 채움
2. **2단계 (숫자형 Scaling)**: Train 데이터 기준 **평균과 표준편차(StandardScaler)**를 구해 나이(`age`) 및 가격(`price`) 스케일 통일
3. **3단계 (범주형 Encoding)**: Train 데이터 기준 최빈값으로 결측치 채운 후 **One-Hot Encoding** (`handle_unknown='ignore'`로 미학습 범주 예외 방지)
4. **4단계 (데이터셋 Concat)**: 전처리된 숫자형 및 범주형 피처를 결합하여 **결측치 0개, 100% 수치형 학습 데이터셋** 완성

---

### 2. Data Leakage 방지 핵심 원칙

| 구분 | 적용 방식 | 이유 및 목적 |
| :--- | :--- | :--- |
| **Train 데이터** | `fit_transform()` | Train 데이터의 분포(중앙값, 평균, 표준편차, 범주 목록)를 학습하고 변환 |
| **Test 데이터** | `transform()`만 적용 | **Train에서 구해진 매개변수만 그대로 사용**하여 평가 (미래 데이터 정보 누수 차단) |

> **⚠️ 전체 데이터에 먼저 `fit`을 적용하면 안 되는 이유**
> Test 데이터의 평균이나 범주 정보가 Train 전처리 과정에 스며들면(Data Leakage), 모델이 시험지 정답 힌트를 알고 학습하는 꼴이 되어 실전(운영 환경) 예측 성능이 크게 저하됩니다.

---

### 3. 최종 전처리 결과 요약

* **학습 피처 (`X_train_prepared`)**: 결측치 0개, 범주형 One-Hot 인코딩 포함 전 피처 수치화 완료
* **평가 피처 (`X_test_prepared`)**: Train 기준으로 구성된 인코더/스케일러 매개변수 기반 변환 완료
* **Pipeline 구조적 확장성**: 수동 전처리와 동일한 로직을 Scikit-Learn `Pipeline` 및 `ColumnTransformer`로 자동화할 수 있음을 검증함

### STEP 6. Baseline과 후보 모델을 준비합니다

In [13]:
from sklearn.dummy import DummyRegressor
from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor

# ---------------------------------------------------------
# STEP 6. Baseline 및 후보 모델 생성 및 학습
# ---------------------------------------------------------
# 1. Baseline Model (단순 평균 예측)
baseline_model = DummyRegressor(strategy="mean")
baseline_model.fit(X_train_prepared, y_train)

# 2. Linear Regression (선형 모델)
lr_model = LinearRegression()
lr_model.fit(X_train_prepared, y_train)

# 3. Random Forest Regressor (비선형/트리 기반 모델)
rf_model = RandomForestRegressor(random_state=42)
rf_model.fit(X_train_prepared, y_train)

# ---------------------------------------------------------
# 모델 준비 완료 결과 확인
# ---------------------------------------------------------
print("## STEP 6. Baseline 및 후보 모델 학습 완료")
print("=" * 60)
print(f"1. Baseline Mean  : {baseline_model.__class__.__name__} (평균값: {y_train.mean():.2f}원)")
print(f"2. Linear Reg.   : {lr_model.__class__.__name__}")
print(f"3. Random Forest : {rf_model.__class__.__name__}")
print("=" * 60)
print("세 가지 모델의 학습이 성공적으로 완료되었습니다.")
print("다음 단계(STEP 7)에서 교차 검증(CV) 및 평가 지표(MAE, RMSE, R²)로 각 모델의 실제 성능을 비교합니다.")

## STEP 6. Baseline 및 후보 모델 학습 완료
1. Baseline Mean  : DummyRegressor (평균값: 836451.48원)
2. Linear Reg.   : LinearRegression
3. Random Forest : RandomForestRegressor
세 가지 모델의 학습이 성공적으로 완료되었습니다.
다음 단계(STEP 7)에서 교차 검증(CV) 및 평가 지표(MAE, RMSE, R²)로 각 모델의 실제 성능을 비교합니다.


### STEP 7. Train 기간에서 TimeSeriesSplit으로 후보를 비교합니다

In [14]:
import os
import numpy as np
import pandas as pd
from sklearn.model_selection import TimeSeriesSplit, cross_validate
from sklearn.dummy import DummyRegressor
from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor

# ---------------------------------------------------------
# 1. 경로 설정 및 파일 저장 준비
# ---------------------------------------------------------
base_dir = os.getcwd()
if "notebooks" in base_dir:
    project_root = os.path.abspath(os.path.join(base_dir, "..", ".."))
else:
    project_root = base_dir

reports_dir = os.path.join(project_root, "reports")
os.makedirs(reports_dir, exist_ok=True)
report_path = os.path.join(reports_dir, "ch09_regression_cv_summary.csv")

# ---------------------------------------------------------
# 2. TimeSeriesSplit 설정 및 후보 모델 정의
# ---------------------------------------------------------
# 시간 순서를 유지하는 5-Fold 시계열 교차 검증
tscv = TimeSeriesSplit(n_splits=5)

models = {
    "Baseline Mean": DummyRegressor(strategy="mean"),
    "Linear Regression": LinearRegression(),
    "Random Forest": RandomForestRegressor(random_state=42)
}

# 평가 평가지표 지정 (MAE, R2)
scoring = {
    "mae": "neg_mean_absolute_error",
    "r2": "r2"
}

# ---------------------------------------------------------
# 3. CV 평가 수행 및 결과 집계
# ---------------------------------------------------------
cv_results_list = []

for name, model in models.items():
    # Train 데이터 내에서만 교차 검증 수행 (Final Test 사용금지)
    scores = cross_validate(model, X_train_prepared, y_train, cv=tscv, scoring=scoring)
    
    # neg_mean_absolute_error를 양수 MAE로 변환
    mae_scores = -scores["test_mae"]
    r2_scores = scores["test_r2"]
    
    cv_results_list.append({
        "model": name,
        "cv_MAE_mean": round(np.mean(mae_scores), 4),
        "cv_MAE_std": round(np.std(mae_scores), 4),
        "cv_R2_mean": round(np.mean(r2_scores), 4)
    })

# DataFrame 변환
cv_summary_df = pd.DataFrame(cv_results_list)

# 비고(Note) 항목 추가 (최종 선정을 위한 로직)
non_baseline_df = cv_summary_df[cv_summary_df["model"] != "Baseline Mean"]
best_model_name = non_baseline_df.loc[non_baseline_df["cv_MAE_mean"].idxmin(), "model"]

notes = []
for idx, row in cv_summary_df.iterrows():
    if row["model"] == "Baseline Mean":
        notes.append("기준선 (Baseline)")
    elif row["model"] == best_model_name:
        notes.append("최종 선정 (Lowest MAE)")
    else:
        notes.append("후보 (탈락)")

cv_summary_df["note"] = notes

# ---------------------------------------------------------
# 4. Evidence 리포트 저장 (reports/ch09_regression_cv_summary.csv)
# ---------------------------------------------------------
cv_summary_df.to_csv(report_path, index=False, encoding="utf-8-sig")

# ---------------------------------------------------------
# 5. Notebook 요약 리포트 출력
# ---------------------------------------------------------
print("## STEP 7. Train 기간 TimeSeriesSplit CV 평가 결과")
print("=" * 70)
display(cv_summary_df)
print("-" * 70)
print(f"• 최종 선택된 모델   : {best_model_name} (Train CV MAE가 더 낮음)")
print(f"• Evidence 저장 경로 : {report_path}")
print("=" * 70)

## STEP 7. Train 기간 TimeSeriesSplit CV 평가 결과


,model,cv_MAE_mean,cv_MAE_std,cv_R2_mean,note
0,Baseline Mean,453552.159,47282.9677,-0.1844,기준선 (Baseline)
1,Linear Regression,680491.627,230696.0215,-13.5476,후보 (탈락)
2,Random Forest,588454.000,112535.8504,-5.6033,최종 선정 (Lowest MAE)


----------------------------------------------------------------------
• 최종 선택된 모델   : Random Forest (Train CV MAE가 더 낮음)
• Evidence 저장 경로 : c:\dev\llm-data-analysis-course\reports\ch09_regression_cv_summary.csv


## STEP 7. Train 기간 TimeSeriesSplit 교차 검증 (CV) 평가

### 1. 모델별 CV 평가 결과 요약

| 모델 | CV MAE 평균 | CV MAE 표준편차 | CV R² 평균 | 비고 |
| :--- | :---: | :---: | :---: | :--- |
| **Baseline Mean** | *측정값* | *측정값* | *측정값* | 기준선 (Baseline) |
| **Linear Regression** | *측정값* | *측정값* | *측정값* | 비베이스라인 후보 |
| **Random Forest** | *측정값* | *측정값* | *측정값* | 비베이스라인 후보 (최종 선정) |

> **📌 참고**: 위 표의 *측정값*은 위 코드 셀 실행 결과 테이블의 숫자를 그대로 기입하시면 됩니다.

---

### 2. 핵심 점검 사항 및 판단 기준
* **Final Test 격리**: Final Test 데이터는 일절 평가에 사용하지 않고, **Train 기간 내 시간 순서(TimeSeriesSplit)**로만 교차 검증을 진행함.
* **최종 후보 선정**: 비베이스라인 모델 중 **Train CV MAE 평균이 더 낮은 모델**을 선택함.
* **안정성(표준편차) 검증**: CV MAE 표준편차(`cv_MAE_std`)를 함께 확인하여 Fold별 성능 흔들림(변동성)이 과도하지 않은지 함께 점검 완료.
* **Evidence 저장 경로**: `reports/ch09_regression_cv_summary.csv`

### STEP 8. Selected Model을 Final Test 전에 고정합니다

## Train CV로 선택한 모델

- **Selected Model**: `Random Forest`
- **선택 기준**: Train 기간 TimeSeriesSplit 5-Fold 교차 검증에서 비베이스라인 후보(Linear Regression, Random Forest) 중 `cv_MAE_mean`이 더 낮은 모델 선택
- **근거가 된 cv_MAE_mean**: 
  - `Random Forest`: **588,454.0001** (선택)
  - `Linear Regression`: 680,491.6272
  - *(참고) Baseline Mean: 453,552.1594*
- **Final Test를 보기 전에 선택했는가**: **`예`** (Test 데이터를 사용하기 전, Train CV 오비어스 규칙에 따라 Random Forest로 사전 확정 완료)

In [15]:
import os
import pandas as pd
from sklearn.ensemble import RandomForestRegressor

# ---------------------------------------------------------
# STEP 8. Selected Model 고정 및 Train 전체 재학습
# ---------------------------------------------------------
# 1. Train CV 결과 기반 Selected Model 명시적 결정 및 고정
selected_model_name = "Random Forest"
selected_model = RandomForestRegressor(random_state=42)

print("## STEP 8. Selected Model 고정 완료")
print("=" * 60)
print(f"• 최종 선택된 모델 (Selected Model) : {selected_model_name}")
print("• 선택 기준                           : Train CV MAE 최저 (588,454.00원)")
print("• Final Test 참조 여부                : 미참조 (0% Leakage)")
print("=" * 60)

# 2. Selected Model을 전체 Train 데이터셋에 대해 재학습 (Re-fit)
# (교차 검증 시에는 Train의 일부 Fold만 학습했으므로, 전체 Train 데이터로 최종 학습)
selected_model.fit(X_train_prepared, y_train)

print(f"• '{selected_model_name}' 모델이 전체 Train 데이터({len(X_train_prepared)}건)로 성공적으로 재학습되었습니다.")
print("• 준비 완료: 이제 고정된 모델로 Final Test 단일 평가(STEP 9)를 진행할 준비가 되었습니다.")
print("=" * 60)

## STEP 8. Selected Model 고정 완료
• 최종 선택된 모델 (Selected Model) : Random Forest
• 선택 기준                           : Train CV MAE 최저 (588,454.00원)
• Final Test 참조 여부                : 미참조 (0% Leakage)
• 'Random Forest' 모델이 전체 Train 데이터(237건)로 성공적으로 재학습되었습니다.
• 준비 완료: 이제 고정된 모델로 Final Test 단일 평가(STEP 9)를 진행할 준비가 되었습니다.


### STEP 9. Final Test에서는 Baseline과 Frozen Model만 평가합니다

In [16]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.dummy import DummyRegressor
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error, root_mean_squared_error, r2_score

# ---------------------------------------------------------
# 1. 경로 및 디렉토리 설정
# ---------------------------------------------------------
base_dir = os.getcwd()
if "notebooks" in base_dir:
    project_root = os.path.abspath(os.path.join(base_dir, "..", ".."))
else:
    project_root = base_dir

reports_dir = os.path.join(project_root, "reports")
images_dir = os.path.join(project_root, "chapter09", "images")

os.makedirs(reports_dir, exist_ok=True)
os.makedirs(images_dir, exist_ok=True)

report_path = os.path.join(reports_dir, "ch09_regression_model_comparison.csv")
image_path = os.path.join(images_dir, "step06_final_metrics.png")

# ---------------------------------------------------------
# 2. Evaluation 대상 모델 준비 (Baseline & Frozen Selected Model)
# ---------------------------------------------------------
eval_models = {
    "Baseline Mean": {
        "role": "baseline",
        "model": DummyRegressor(strategy="mean")
    },
    "Frozen Selected Model": {
        "role": "selectedbytrain_cv",
        "model": RandomForestRegressor(random_state=42)
    }
}

# ---------------------------------------------------------
# 3. Train 재학습 및 Final Test 평가 수행
# ---------------------------------------------------------
results = []

for name, meta in eval_models.items():
    model = meta["model"]
    role = meta["role"]
    
    # Train 데이터 전체로 학습
    model.fit(X_train_prepared, y_train)
    
    # 예측 수행
    train_pred = model.predict(X_train_prepared)
    test_pred = model.predict(X_test_prepared)
    
    # 지표 계산
    tr_mae = mean_absolute_error(y_train, train_pred)
    te_mae = mean_absolute_error(y_test, test_pred)
    te_rmse = root_mean_squared_error(y_test, test_pred)
    te_r2 = r2_score(y_test, test_pred)
    
    results.append({
        "model": name,
        "selection_role": role,
        "train_MAE": round(tr_mae, 4),
        "test_MAE": round(te_mae, 4),
        "test_RMSE": round(te_rmse, 4),
        "test_R2": round(te_r2, 4)
    })

# DataFrame 생성
comparison_df = pd.DataFrame(results)

# Baseline 대비 MAE 개선율 계산 (%)
baseline_test_mae = comparison_df.loc[comparison_df["model"] == "Baseline Mean", "test_MAE"].values[0]

improvements = []
for idx, row in comparison_df.iterrows():
    if row["model"] == "Baseline Mean":
        improvements.append(0.0)
    else:
        # (Baseline MAE - Model MAE) / Baseline MAE * 100
        imp = ((baseline_test_mae - row["test_MAE"]) / baseline_test_mae) * 100
        improvements.append(round(imp, 2))

comparison_df["MAE_improvement_vs_baseline_pct"] = improvements

# ---------------------------------------------------------
# 4. Evidence CSV 저장 (reports/ch09_regression_model_comparison.csv)
# ---------------------------------------------------------
comparison_df.to_csv(report_path, index=False, encoding="utf-8-sig")

# ---------------------------------------------------------
# 5. Evidence 이미지 생성 및 저장 (chapter09/images/step06_final_metrics.png)
# ---------------------------------------------------------
fig, ax = plt.subplots(figsize=(8, 4), dpi=150)

models_list = comparison_df["model"].tolist()
mae_values = comparison_df["test_MAE"].tolist()
colors = ["#7f7f7f" if r == "baseline" else "#d62728" for r in comparison_df["selection_role"]]

bars = ax.bar(models_list, mae_values, color=colors, width=0.4)

ax.set_title("Final Test MAE Comparison (Baseline vs Frozen Selected Model)", fontsize=12, fontweight="bold", pad=15)
ax.set_ylabel("Test MAE (Lower is better)", fontsize=10)
ax.set_ylim(0, max(mae_values) * 1.25)

# 수치 라벨링
for bar in bars:
    height = bar.get_height()
    ax.annotate(f"{height:,.1f}",
                xy=(bar.get_x() + bar.get_width() / 2, height),
                xytext=(0, 5),
                textcoords="offset points",
                ha="center", va="bottom", fontsize=10, fontweight="bold")

plt.tight_layout()
plt.savefig(image_path)
plt.close()

# ---------------------------------------------------------
# 6. 결과 출력
# ---------------------------------------------------------
print("## STEP 9. Final Test 평가 결과")
print("=" * 80)
display(comparison_df)
print("-" * 80)
print(f"• Evidence CSV 저장 완료   : {report_path}")
print(f"• Evidence 이미지 저장 완료 : {image_path}")
print("=" * 80)

## STEP 9. Final Test 평가 결과


,model,selection_role,train_MAE,test_MAE,test_RMSE,test_R2,MAE_improvement_vs_baseline_pct
0,Baseline Mean,baseline,488976.5529,459236.5682,552306.2554,-0.0005,0.00
1,Frozen Selected Model,selectedbytrain_cv,206381.3502,433575.5000,546066.4684,0.0220,5.59


--------------------------------------------------------------------------------
• Evidence CSV 저장 완료   : c:\dev\llm-data-analysis-course\reports\ch09_regression_model_comparison.csv
• Evidence 이미지 저장 완료 : c:\dev\llm-data-analysis-course\chapter09\images\step06_final_metrics.png


## STEP 9. Final Test 평가 및 최종 회귀 파이프라인 마무리

### 1. Final Test 지표 비교 표

| 모델 | selection role | Train MAE | Test MAE | Test RMSE | Test R² | Baseline 대비 MAE 개선율 |
| :--- | :--- | :---: | :---: | :---: | :---: | :---: |
| **Baseline Mean** | `baseline` | 488,976.55 | 459,236.57 | 552,306.26 | -0.0005 | 0.00% |
| **Frozen Selected Model** | `selectedbytrain_cv` | 206,381.35 | **433,575.50** | **546,066.47** | **0.0220** | **+5.59%** |

---

### 2. 최종 평가 및 인사이트 요약
1. **엄격한 평가 규칙 준수**: Test Leakage 방지를 위해 Train CV 결과만으로 `Random Forest`를 사전 고정하였고, Final Test 데이터는 오직 단 1회의 최종 검증용으로만 사용함.
2. **성능 개선 확인**: Final Test 결과, 고정된 Random Forest 모델이 단순 평균(Baseline) 대비 **MAE를 약 25,661원(5.59%) 감소**시켰으며, $R^2$ 역시 양수(0.0220)로 반전됨.
3. **향후 모델 고도화 방안**:
   * **과적합 완화**: `max_depth`, `min_samples_leaf` 등의 하이퍼파라미터 튜닝을 통한 모델 규제 강화
   * **피처 엔지니어링**: 단순 고객/상품 기본 정보 외에 '주문당 수량(`quantity`)', '고객별 과거 총 주문 건수/금액' 등 **설명력이 높은 도메인 피처 추가 생성** 필요

---

### 3. 생성된 자동 Evidence
- **CSV 리포트**: `reports/ch09_regression_model_comparison.csv`
- **시각화 이미지**: `chapter09/images/step06_final_metrics.png`

### STEP 10. MAE, RMSE, R²를 함께 해석합니다

## STEP 10. Final Test 지표 해석 및 최종 판단

### 1. 결과 관찰
* **MAE (Mean Absolute Error)**: **`433,575.50원`**
  * 모델의 예측값이 실제 주문 금액과 평균적으로 약 **43만 3,575원** 차이가 납니다.
* **RMSE (Root Mean Squared Error)**: **`546,066.47원`**
  * MAE보다 약 **11만 2,491원 더 높게** 집계되었습니다.
* **R² (Coefficient of Determination)**: **`0.0220`**
  * 모델이 주문 금액 변동성의 약 **2.2%**만을 설명하고 있습니다.
* **Baseline 대비 개선율**: **`+5.59%`**
  * 단순 평균 예측 모델(Baseline MAE: 459,236원) 대비 오차를 약 **25,661원 단축**했습니다.

---

### 2. 나의 해석과 판단
1. **큰 오차(Outlier)의 영향 존재**:
   * RMSE(54.6만 원)가 MAE(43.3만 원)보다 눈에 띄게 큽니다. 오차를 제곱하여 가중치를 두는 RMSE의 특성상, **대형 주문(고액 결제 건)에서 발생한 심각한 예측 오차가 전체 평가지표를 끌어올리고 있음**을 의미합니다.
2. **미미한 설명력과 한계**:
   * R²가 음수(-5.60)였던 Train CV에 비해서는 양수(0.0220)로 반전되며 평균 예측을 근소하게 앞섰지만, **2.2%라는 설명력은 사실상 패턴을 거의 잡지 못하고 있는 상태**입니다.
3. **엄격한 평가 체계의 성공**:
   * 지표 점수 자체는 낮지만, Test Leakage 없이 Train CV만으로 모델을 고정하고 평가한 Validation 프로세스는 이상 없이 성공적으로 작동했습니다.

---

### 3. 업무·분석적 의미
* **현 상태로 실전 배치(Production Deployment) 불가**:
  * 평균 43만 원의 예측 오차는 재고 수량 예측, 매출 추정, 마케팅 예산 편성 등 실제 비즈니스 의사결정에 활용하기에 지나치게 큽니다.
* **의사결정 기준 제시**:
  * 단순 고객 정보(`age`, `gender`, `city`)와 단가(`price`) 정보만으로는 고객의 **"총 주문 금액(`order_total`)"**을 예측할 수 없음을 데이터와 수치 증적으로 입증했습니다.

---

### 4. 현재 한계 및 차기 개선 과제
1. **주요 입력 피처(Feature)의 부재**:
   * 주문 금액 결정을 좌우하는 **'구매 수량(`quantity`)'**, **'장바구니 담긴 총 상품 수'**, **'고객의 과거 누적 결제액'** 등의 핵심 파생 피처가 누락되어 있습니다.
2. **이상치 및 고액 주문 처리 필요**:
   * RMSE 상승의 주원인인 고액 주문 건에 대해 타겟 로그 변환(Log Transformation)이나 이상치 제거/분리 모델링이 필요합니다.
3. **과적합(Overfitting) 관리**:
   * Train MAE(20.6만 원) 대비 Test MAE(43.3만 원)의 오차가 여전히 크므로, Random Forest의 트리를 규제(`max_depth` 제한 등)하거나 차원 축소를 진행해야 합니다.

### STEP 11. 대표 오차 사례를 진단합니다

STEP 11 조치 내용 및 이유

어떤 조치를 하는가?
Final Test 결과에서 오차가 가장 큰 상위 2~3개 주문을 뽑아 실제값, 예측값, 잔차(실제-예측)를 확인하고, 왜 크게 틀렸는지 원인 가설을 진단 및 기록합니다.

왜 하는가?
전체 평균 점수(MAE)만 보면 모델이 '어디서, 어떻게' 틀리는지 알 수 없기 때문입니다.

직관적 사례
예를 들어 평소 1~2만 원을 쓰던 쇼핑몰에서 한 고객이 100만 원어치 단체 주문을 한 경우, 모델은 과거 평균대로 3만 원으로 예측하여 97만 원의 거대한 오차가 발생합니다. 이처럼 단체 주문, 고가 프리미엄 상품 구매 등 '모델이 미처 반영하지 못한 특수한 상황'을 찾아내야, 향후 '단체 주문 여부' 같은 새로운 피처를 추가하여 모델을 개선할 수 있습니다.

In [17]:
import os
import pandas as pd

# ---------------------------------------------------------
# STEP 11. 대표 오차 사례 진단 데이터 생성 및 저장
# ---------------------------------------------------------
# Test 데이터 예측 수행 및 잔차 계산
test_predictions = selected_model.predict(X_test_prepared)

diag_df = test_df[["order_id", "customer_id", "order_date", "order_total"]].copy()
diag_df = diag_df.rename(columns={"order_total": "actual_order_total"})
diag_df["predicted_order_total"] = test_predictions

# 잔차(residual) 및 절대 오차(abs_error) 계산
diag_df["residual"] = diag_df["actual_order_total"] - diag_df["predicted_order_total"]
diag_df["abs_error"] = diag_df["residual"].abs()

# 절대 오차 기준 내림차순 정렬
diag_df = diag_df.sort_values(by="abs_error", ascending=False).reset_index(drop=True)

# ---------------------------------------------------------
# 내부 진단 파일 저장 (order_id 포함)
# ---------------------------------------------------------
base_dir = os.getcwd()
project_root = os.path.abspath(os.path.join(base_dir, "..", "..")) if "notebooks" in base_dir else base_dir
reports_dir = os.path.join(project_root, "reports")
os.makedirs(reports_dir, exist_ok=True)

internal_report_path = os.path.join(reports_dir, "ch09_regression_predictions_internal.csv")
diag_df.to_csv(internal_report_path, index=False, encoding="utf-8-sig")

# ---------------------------------------------------------
# 상위 3개 대표 오차 사례 출력
# ---------------------------------------------------------
print("## STEP 11. 대표 오차 사례 진단 (Top 3 Worst Cases)")
print("=" * 80)
print(f"• 내부 진단 파일 저장 완료: {internal_report_path}")
print("-" * 80)

for idx in range(min(3, len(diag_df))):
    row = diag_df.iloc[idx]
    direction = "실제보다 낮게 예측 (과소예측)" if row["residual"] > 0 else "실제보다 높게 예측 (과대예측)"
    print(f"### 사례 {idx + 1} (Order ID: {row['order_id']})")
    print(f"• 실제값 (actual_order_total)   : {row['actual_order_total']:,.0f}원")
    print(f"• 예측값 (predicted_order_total): {row['predicted_order_total']:,.0f}원")
    print(f"• 잔차 (residual)               : {row['residual']:,.0f}원 ({direction})")
    print(f"• 절대 오차 (abs_error)         : {row['abs_error']:,.0f}원")
    print("-" * 80)

## STEP 11. 대표 오차 사례 진단 (Top 3 Worst Cases)
• 내부 진단 파일 저장 완료: c:\dev\llm-data-analysis-course\reports\ch09_regression_predictions_internal.csv
--------------------------------------------------------------------------------
### 사례 1 (Order ID: 287)
• 실제값 (actual_order_total)   : 1,714,000원
• 예측값 (predicted_order_total): 3,036,750원
• 잔차 (residual)               : -1,322,750원 (실제보다 높게 예측 (과대예측))
• 절대 오차 (abs_error)         : 1,322,750원
--------------------------------------------------------------------------------
### 사례 2 (Order ID: 251)
• 실제값 (actual_order_total)   : 160,000원
• 예측값 (predicted_order_total): 1,444,640원
• 잔차 (residual)               : -1,284,640원 (실제보다 높게 예측 (과대예측))
• 절대 오차 (abs_error)         : 1,284,640원
--------------------------------------------------------------------------------
### 사례 3 (Order ID: 225)
• 실제값 (actual_order_total)   : 1,878,000원
• 예측값 (predicted_order_total): 593,910원
• 잔차 (residual)               : 1,284,090원 (실제보다 낮게 예측 (과소예측))
• 절대 오차 (abs_error

### STEP 11 대표 오차 사례 분석 결과 및 진단서

#### 1. 사례별 관찰 및 가설

* **사례 1 (Order ID: 287)**
  * **실제값**: 1,714,000원 | **예측값**: 3,036,750원 | **잔차**: -1,322,750원 (과대예측)
  * **관찰**: 모델이 실제 결제액보다 약 132만 원 높게 예측함.
  * **가설**: 고가 상품 품목이 포함되어 예측값이 상승했으나, 실제 구매 수량이 적었거나 대형 할인이 적용되었을 가능성이 큼.

* **사례 2 (Order ID: 251)**
  * **실제값**: 160,000원 | **예측값**: 1,444,640원 | **잔차**: -1,284,640원 (과대예측)
  * **관찰**: 16만 원 소액 결제건을 144만 원으로 약 9배 높게 뻥튀기 예측함.
  * **가설**: 단가(`price`) 정보에 의존하여 높은 예측값을 출력했으나, 실제 수량은 1개에 불과했을 것임.

* **사례 3 (Order ID: 225)**
  * **실제값**: 1,878,000원 | **예측값**: 593,910원 | **잔차**: +1,284,090원 (과소예측)
  * **관찰**: 187만 원 대형 결제건을 59만 원으로 대폭 낮게 예측함.
  * **가설**: 낮은 단가의 상품을 대량 구매(단체 주문)한 건이나, '수량' 피처가 없어 일반 소액 주문으로 오인함.

---

#### 2. 진단 결론 및 개선 방향
* **핵심 원인**: 상위 오차 사례 모두 **'구매 수량(`quantity`)' 및 '할인 금액' 정보의 부재**로 인해 실제 주문 규모를 왜곡하여 예측함.
* **차기 피처 엔지니어링 1순위**: `quantity`(주문 수량) 피처 및 `고객별 과거 평균 구매 수량` 피처 추가 생성 필요.

### STEP 12. 자동 Validation Evidence를 확인합니다

무엇을 위한 작업인가?
중요한 시험을 치른 후 답안지 제출 전 '규정 위반이나 치명적인 데이터 누수(Data Leakage)가 없었는지' 시스템이 자동으로 검증하여 결과의 신뢰성을 보장하는 작업입니다.

직관적 비유
건축물이 아무리 화려해도 '소방·안전 검사(Validation)'를 통과하지 못하면 준공 허가가 안 나는 것과 같습니다.

In [18]:
import os
import pandas as pd

# ---------------------------------------------------------
# 1. Validation 결과 파일 경로 설정 및 확인
# ---------------------------------------------------------
base_dir = os.getcwd()
if "notebooks" in base_dir:
    project_root = os.path.abspath(os.path.join(base_dir, "..", ".."))
else:
    project_root = base_dir

reports_dir = os.path.join(project_root, "reports")
os.makedirs(reports_dir, exist_ok=True)
validation_path = os.path.join(reports_dir, "ch09_regression_validation.csv")

# ---------------------------------------------------------
# 2. 파이프라인 무결성 자체 검사 (Validation Rule Engine)
# ---------------------------------------------------------
validation_checks = []

# ① forbidden_feature_overlap (Train과 Test간 중복 행/특정 피처 오버랩 검사)
train_indices = set(X_train_prepared.index)
test_indices = set(X_test_prepared.index)
overlap_count = len(train_indices.intersection(test_indices))
validation_checks.append({
    "check": "forbidden_feature_overlap",
    "value": f"Overlap rows: {overlap_count}",
    "status": "PASS" if overlap_count == 0 else "FAIL"
})

# ② strict_train_before_test (시간 순서 엄격 준수 여부 검사)
train_max_date = train_df["order_date"].max()
test_min_date = test_df["order_date"].min()
is_strict_time = train_max_date <= test_min_date
validation_checks.append({
    "check": "strict_train_before_test",
    "value": f"Train Max ({train_max_date.strftime('%Y-%m-%d')}) <= Test Min ({test_min_date.strftime('%Y-%m-%d')})",
    "status": "PASS" if is_strict_time else "FAIL"
})

# ③ selected_model_exists_in_train_cv (Train CV 결과 파일 존재 및 Selected Model 검증)
cv_summary_path = os.path.join(reports_dir, "ch09_regression_cv_summary.csv")
cv_exists = os.path.exists(cv_summary_path)
validation_checks.append({
    "check": "selected_model_exists_in_train_cv",
    "value": f"CV Report Exists: {cv_exists}",
    "status": "PASS" if cv_exists else "FAIL"
})

# ④ final_test_contains_baseline (최종 비교 파일에 Baseline 포함 여부)
final_report_path = os.path.join(reports_dir, "ch09_regression_model_comparison.csv")
final_report_exists = os.path.exists(final_report_path)
has_baseline = False
has_frozen = False

if final_report_exists:
    final_df = pd.read_csv(final_report_path)
    has_baseline = "Baseline Mean" in final_df["model"].values
    has_frozen = "Frozen Selected Model" in final_df["model"].values

validation_checks.append({
    "check": "final_test_contains_baseline",
    "value": f"Baseline Mean In Final: {has_baseline}",
    "status": "PASS" if has_baseline else "FAIL"
})

# ⑤ final_test_contains_frozen_selected_model (최종 비교 파일에 Frozen Model 포함 여부)
validation_checks.append({
    "check": "final_test_contains_frozen_selected_model",
    "value": f"Frozen Model In Final: {has_frozen}",
    "status": "PASS" if has_frozen else "FAIL"
})

# ⑥ test_rows_for_r2 (R2 계산에 충분한 Test 샘플 수 검사: 최소 2개 이상)
test_rows_count = len(y_test)
validation_checks.append({
    "check": "test_rows_for_r2",
    "value": f"Test Rows: {test_rows_count}",
    "status": "PASS" if test_rows_count >= 2 else "FAIL"
})

# ---------------------------------------------------------
# 3. Validation DataFrame 변환 및 파일 저장
# ---------------------------------------------------------
val_df = pd.DataFrame(validation_checks)
val_df.to_csv(validation_path, index=False, encoding="utf-8-sig")

# ---------------------------------------------------------
# 4. 검증 결과 출력 및 전체 통과 여부 판정
# ---------------------------------------------------------
print("## STEP 12. 자동 Validation Evidence 검증 결과")
print("=" * 70)
display(val_df)
print("-" * 70)

all_pass = (val_df["status"] == "PASS").all()

if all_pass:
    print("✅ [ALL PASS] 모든 품질 검사를 완벽하게 통과했습니다!")
    print("   Chapter 09 머신러닝 회귀 분석 파이프라인이 정상적으로 완료 상태 처리되었습니다.")
else:
    print("❌ [FAIL DETECTED] 품질 검사 항목 중 FAIL이 존재합니다.")
    print("   위의 FAIL 항목 원인을 수정한 후 파이프라인을 재실행해 주세요.")

print(f"• Evidence 저장 경로: {validation_path}")
print("=" * 70)

## STEP 12. 자동 Validation Evidence 검증 결과


,check,value,status
0,forbidden_feature_overlap,Overlap rows: 0,PASS
1,strict_train_before_test,Train Max (2026-06-02) <= Test Min (2026-06-04),PASS
2,selected_model_exists_in_train_cv,CV Report Exists: True,PASS
3,final_test_contains_baseline,Baseline Mean In Final: True,PASS
4,final_test_contains_frozen_selected_model,Frozen Model In Final: True,PASS
5,test_rows_for_r2,Test Rows: 60,PASS


----------------------------------------------------------------------
✅ [ALL PASS] 모든 품질 검사를 완벽하게 통과했습니다!
   Chapter 09 머신러닝 회귀 분석 파이프라인이 정상적으로 완료 상태 처리되었습니다.
• Evidence 저장 경로: c:\dev\llm-data-analysis-course\reports\ch09_regression_validation.csv


## STEP 12. 자동 Validation Evidence 최종 검증 기록

### 1. 무결성 검사 결과 요약 (ALL PASS)

| check | value | status | 비고 / 검증 목적 |
| :--- | :--- | :---: | :--- |
| **forbidden_feature_overlap** | Overlap rows: 0 | **PASS** | Train과 Test 간 인덱스/행 중복 없음 (Leakage 차단) |
| **strict_train_before_test** | Train Max (2026-06-02) <= Test Min (2026-06-04) | **PASS** | 미래 데이터 누수 없는 엄격한 시계열 분할 준수 |
| **selected_model_exists_in_train_cv** | CV Report Exists: True | **PASS** | Final Test 전 Train CV 결과 기반 모델 선정 증적 존재 |
| **final_test_contains_baseline** | Baseline Mean In Final: True | **PASS** | Final Test 평가 시 단순 평균(Baseline) 대조군 포함 |
| **final_test_contains_frozen_selected_model** | Frozen Model In Final: True | **PASS** | CV 기반 고정 모델(Random Forest) 단일 평가 완료 |
| **test_rows_for_r2** | Test Rows: 60 | **PASS** | $R^2$ 신뢰도 확보를 위한 충분한 평가 데이터 수 확보 (60건) |

---

### 2. Chapter 09 실습 최종 마감 요약 및 레슨

1. **엄격한 Validation 체계 구축**:
   * Test 데이터의 정답 힌트가 Train 과정에 스며들지 않도록 `fit_transform()`과 `transform()`을 명확히 분리함.
   * Final Test를 진행하기 전 Train CV 결과만으로 `Random Forest`를 고정(Frozen)하여 모델 선택 과정의 불확실성과 평가 누수를 원천 봉쇄함.
2. **성능 평가 및 비즈니스 인사이트**:
   * Final Test 결과, 고정된 Random Forest 모델이 **Baseline 대비 MAE 5.59% 개선** 및 **양수 $R^2$ (0.0220) 달성**을 기록함.
   * 오차 진단(STEP 11)을 통해 현재 입력 피처(`age`, `price` 등)만으로는 '구매 수량(`quantity`)' 변화나 단체 주문 같은 대형 결제건을 설명하기 어렵다는 핵심 한계를 데이터 수치로 입증함.
3. **최종 결론**:
   * 모든 자동 검증 규칙(Validation Rule)을 **ALL PASS**로 통과하여 Chapter 09 머신러닝 회귀 분석 파이프라인의 구축 및 검증을 완료 상태로 처리함.

---

### 📁 생성된 증적(Evidence) 파일 목록
* `reports/ch09_regression_cv_summary.csv` (Train CV 평가 리포트)
* `reports/ch09_regression_model_comparison.csv` (Final Test 모델 비교 리포트)
* `reports/ch09_regression_predictions_internal.csv` (내부 오차 진단 데이터)
* `reports/ch09_regression_validation.csv` (파이프라인 무결성 검증 리포트)
* `chapter09/images/step06_final_metrics.png` (Final Test MAE 시각화 차트)

### STEP 13. Evidence와 공개 범위를 구분합니다

## STEP 13. Evidence와 공개 범위 구분 및 검토

### 1. 공개 범위 분류 요약

* **공개 가능한 Evidence 및 보고서**:
  * `reports/ch09_regression_split_summary.csv`
  * `reports/ch09_regression_feature_audit.csv`
  * `reports/ch09_regression_cv_summary.csv`
  * `reports/ch09_regression_model_comparison.csv`
  * `reports/ch09_regression_validation.csv`
  * `reports/ch09_regression_checklist.csv`
  * `reports/ch09_regression_report.md`
  * `reports/figures/ch09_actual_vs_predicted.png`
  * `reports/figures/ch09_residual_histogram.png`

* **Internal로만 유지할 파일 (비공개)**:
  * `reports/ch09_regression_model_data_internal.csv`
  * `reports/ch09_regression_predictions_internal.csv`

---

### 2. Internal로 구분한 이유 및 보안 가이드라인
1. **식별자(Identifier) 누출 방지**:
   * Internal 파일에는 행 단위 데이터의 원본 식별자인 `order_id`, `customer_id` 등의 비즈니스 키가 들어있어 고객의 실제 결제 내역을 역추적할 위험이 있습니다.
2. **분석 목적과의 부합성**:
   * 외부 공유용 모델 검증 보고서에서는 **전체적인 모델의 성능(MAE, RMSE, R²)과 패턴 분석 그래프**만으로도 검증 목적을 충분히 달성할 수 있습니다.
   * 불필요한 개별 식별자는 제거하고, 오차 진단 목적의 세부 데이터는 사내 내부 저장소(`Internal`)에만 격리 보관합니다.
3. **거버넌스 원칙**:
   * 모델 성능 검증(Model Validation)과 결과물 공개 범위 검토(Public Disclosure Review)는 별개의 독립된 검증 항목으로 관리합니다.

### STEP 14. 전체 파이프라인을 처음부터 다시 실행합니다

### 1. 실행 및 검증 체크리스트

* **입력 준비 성공 여부**: **`성공`** (Raw 데이터 로드 및 Foreign Key 검증 이상 없음)
* **회귀 파이프라인 셀 전체 실행 성공 여부**: **`성공`** (Kernel Restart 후 단일 흐름으로 오류 없이 완료)
* **Train CV Selected Model**: **`Random Forest`** (MAE 588,454원 기준 사전 고정)
* **Final Validation 전체 PASS 여부**: **`ALL PASS`** (6/6 무결성 규칙 완벽 통과)
* **Notebook 셀 재실행 결과의 일관성**: **`완벽 일치`** (동일 시드 `random_state=42` 고정으로 매번 동일한 MAE 433,575원 및 R² 0.0220 산출)

---

### 2. 재실행 검증 결론
- **메모리 의존성 제거**: 변수 재할당이나 이전 셀 오염 없이 완전한 순차 실행(Top-to-Bottom Execution) 성공.
- **재현성(Reproducibility) 확보**: 별도의 외부 `.py` 스크립트 파일 없이도 주피터 노트북 자체만으로 100% 동일한 실험 결과 및 Evidence 파일들을 재생성함을 최종 입증함.

### STEP 15. 모델 사용 가능성을 판단하고 제출합니다

### 최종 사용 판단
* **판단 결과**: **`현재 데이터로는 사용 보류`**

---

### 판단 근거
1. **미흡한 설명력 및 높은 예측 오차**:
   * Final Test MAE가 약 **43만 3,575원**으로, 예측값이 실제 금액과 평균 43만 원 이상 차이가 납니다.
   * $R^2$가 **0.0220**으로, 모델이 타겟 변동성의 단 2.2%만 설명하고 있어 실전 비즈니스 예측 모델로서의 기능이 불가능합니다.
2. **Train CV 대비 과적합(Overfitting) 존재**:
   * Train MAE(20.6만 원)와 Test MAE(43.3만 원) 간 약 22.7만 원의 심각한 성능 격차가 존재합니다.
3. **규정 및 무결성 검증은 통과(ALL PASS)**:
   * 시간 순서 분할, Data Leakage 방지, 자동 검증 등 절차적 무결성은 완벽히 통과했으나, **모델 자체의 절대적인 성능 한계**로 인해 사용 보류를 결정합니다.

---

### 업무적 위험
* 현재 모델을 재고 관리, 매출 예측, 마케팅 예산 편성 등에 그대로 적용할 경우, **평균 43만 원 이상의 거대한 오차로 인해 심각한 재무적 손실이나 수량 예측 실패**를 초과 초래할 위험이 있습니다.

---

### 현재 데이터의 한계
* 입력 피처(`age`, `price`, `city` 등)에 **'주문 수량(`quantity`)'**, **'장바구니 총 상품 종류'**, **'할인 쿠폰 금액'** 등 주문 총액을 직접적으로 결정짓는 핵심 변수가 누락되어 있습니다.

---

### 다음 개선 우선순위
1. **피처 엔지니어링 (1순위)**: `quantity`(주문 수량) 피처 및 `고객별 과거 누적 구매액` 등 고설명력 파생 피처 추가
2. **고액 주문 이상치 처리**: RMSE를 끌어올리는 고액 결제건 처리를 위한 타겟 로그 변환($Log Transformation$) 또는 모델 분리
3. **모델 규제(Regularization) 강화**: Random Forest의 `max_depth` 제한을 통한 과적합 완화 및 하이퍼파라미터 튜닝

In [21]:
import pandas as pd

# 1. 이미 전처리되어 있는 X_test_prepared 데이터에서 상위 3건 추출
sample_prepared = X_test_prepared[:3]

# 2. AI 모델로 예측
sample_preds = selected_model.predict(sample_prepared)

# 3. 실제 정답(y_test)과 AI 예측값 비교 출력
sample_actuals = y_test.iloc[:3].values

print("## 🤖 실제 데이터(Test Data) 3건에 대한 AI 예측 결과 비교")
print("=" * 65)
for i in range(3):
    actual = sample_actuals[i]
    pred = sample_preds[i]
    diff = actual - pred
    print(f"샘플 {i+1}:")
    print(f"  • 실제 결제 금액 : {actual:,.0f}원")
    print(f"  • AI가 예측한 금액: {pred:,.0f}원")
    print(f"  • 오차 (실제-예측): {diff:,.0f}원")
    print("-" * 65)

## 🤖 실제 데이터(Test Data) 3건에 대한 AI 예측 결과 비교
샘플 1:
  • 실제 결제 금액 : 969,000원
  • AI가 예측한 금액: 741,160원
  • 오차 (실제-예측): 227,840원
-----------------------------------------------------------------
샘플 2:
  • 실제 결제 금액 : 1,001,000원
  • AI가 예측한 금액: 856,790원
  • 오차 (실제-예측): 144,210원
-----------------------------------------------------------------
샘플 3:
  • 실제 결제 금액 : 1,339,000원
  • AI가 예측한 금액: 814,600원
  • 오차 (실제-예측): 524,400원
-----------------------------------------------------------------


### STEP 16. 현재 예측 요구사항의 한계를 다시 확인합니다

## 현재 예측 요구사항의 한계

### 현재 Target
* **Target 정의**: `order_total` (주문건별 총 금액)
* **산출 공식**: 
  * $\text{line\_total} = \text{quantity} \times \text{unit\_price}$
  * $\text{order\_total} = \sum \text{line\_total}$ (동일 `order_id` 기준)

### 이 값을 정확히 계산할 수 있는 시점
* **결제/주문 완료 시점**: 고객이 장바구니에 상품을 담고 주문을 확정하는 순간, 각 품목의 수량(`quantity`)과 단가(`unit_price`)가 즉시 확정됩니다.
* **산술적 계산 가능**: 데이터베이스에 주문 내역이 저장되는 순간 `order_total`은 단 1ms의 단순 사칙연산만으로 오차 범위 0%의 정확한 값을 얻을 수 있습니다.

### 현재 회귀 예측이 교육용으로는 의미가 있지만 실제 업무 예측으로는 제한적인 이유
* **교육적 의미**:
  * 데이터 수집부터 평가까지 회귀분석의 전체 End-to-End 프로세스 학습
  * 미래 시점의 정보가 유입되는 Feature Leakage 방지 개념 체득
  * TimeSeriesSplit 기반의 시간 순서 평가 기법 이해
  * Baseline 모델과 고급 모델 간의 성능 비교 및 MAE·RMSE·$R^2$·잔차 분석 역량 확보
* **실제 업무 관점의 한계 (사용 이유 부재)**:
  * 머신러닝 모델은 **'불확실성이 존재하는 미래의 값'**이나 **'측정하기 어려운 숨겨진 값'**을 추정하기 위해 사용합니다.
  * 이미 100% 확정된 산술 공식이 존재하는 상황에서, 오차가 발생하는 머신러닝 알고리즘으로 `order_total`을 예측하는 것은 비효율적이며 실제 비즈니스 가치를 창출하지 못합니다.

### 더 의미 있는 예측 문제라면 무엇이 달라져야 하는가
실제 업무에서 머신러닝의 가치를 발휘하려면 **'주문 정보가 확정되기 전 시점'**으로 예측 시점을 앞당기거나 **'미래의 행동 및 가치'**를 예측하도록 문제를 재정의해야 합니다.

1. **예측 시점의 변화 (시점 앞당기기)**
   * **주문 발생 전 예측**: 고객의 방문/장바구니 담기 시점에 해당 주문이 최종 결제로 이어질지, 혹은 최종 결제 금액이 얼마가 될지 예측
2. **Target 대상의 변화 (미래 가치 예측)**
   * **고객 생애 가치 (LTV / 30일 후 재구매액)**: 특정 고객이 향후 1달 또는 1년 동안 총 얼마를 지출할지 예측
   * **수요 예측 (Demand Forecasting)**: 내일 또는 다음 주 특정 상품이 총 몇 개 팔릴지 예측하여 재고 관리 최적화
   * **환불/취소 가능성 예측**: 해당 주문이 추후 취소되거나 환불될 확률/금액 예측

### STEP 17. 실제로 의미 있는 새로운 회귀 예측 요구사항을 정의합니다

## STEP 17. 비즈니스 관점의 새로운 회귀 예측 요구사항 정의

기존 `order_total` 예측 문제의 한계(결제 시점 즉시 100% 계산 가능)를 극복하고, 실제 비즈니스 의사결정에 기여할 수 있는 새로운 회귀 예측 문제를 아래와 같이 정의합니다.

---

### 1. 선택한 새로운 예측 문제
* **문제 정의**: 과거 판매 이력 및 시계열 패턴 데이터를 기반으로 **'다음 달 카테고리별 총 완료 매출(Completed Revenue)'**을 미리 예측하는 문제

### 2. Target 및 예측 시점 정의
* **Target (y)**: `next_month_category_revenue` (숫자형 연속값, Continuous Numeric)
  * 특정 카테고리의 다음 달 완료(Completed) 상태 주문 총 금액 sum(`order_total`)
* **예측 시점 (Cut-off Date)**: 매월 말일 23시 59분 (예: 10월 31일)
* **Target 관측 기간**: 예측 시점 이후 1달간 (예: 11월 1일 ~ 11월 30일)

### 3. Feature 구성 원칙 (Data Leakage 완전 차단)
* **사용 가능 데이터 범위**: 예측 시점(매월 말일) **이전**에 발생하여 이미 확정된 데이터만 활용
* **주요 Feature 예시**:
  * **과거 매출 지표**: 최근 1개월/3개월/6개월 해당 카테고리 총 매출액 및 성장률
  * **주문/고객 지표**: 최근 1개월 카테고리별 구매 고객 수, 평균 주문 단가(AOV), 재구매율
  * **계절성/시계열 지표**: 전년 동월 매출액, 월별 계절성 인덱스, 최근 주차별 매출 추세 기울기
* **누수 방지 규칙**: 예측 대상 기간(다음 달)의 주문 내역, 수량, 할인율 등 미래 정보는 Feature 생성에 절대 포함하지 않음

### 4. 예측 결과의 비즈니스 활용 목적 (Business Value)
1. **재고 및 발주 최적화**: 카테고리별 예상 수요에 맞춰 사전 재고를 확보함으로써 품절(Lost Sales) 및 과다 재고 비용 최소화
2. **프로모션 및 마케팅 예산 배분**: 매출 감소가 예상되는 카테고리에 타겟 프로모션을 사전 기획하고 예산을 효율적으로 집행
3. **매출 목표 및 예산 수립**: 사업부별 차기 월 매출 목표 설정 및 이익률 추정을 위한 객관적 기준선(Baseline)으로 활용

### STEP 18. 최종 과제의 Prediction Contract를 작성합니다

## 최종 과제 Prediction Contract

### 예측 목적
* **의사결정 목표**: 다음 달 카테고리별 예상 매출을 사전 파악하여 **재고 발주 최적화**, **마케팅 예산 배분**, **매출 목표 수립** 등 비즈니스 계획 실행
* **다섯 질문 답안**:
  1. **무엇을 예측하는가?**: 다음 달 카테고리별 완료된 주문의 총 매출액 (`total_sales`)
  2. **언제 예측하는가?**: 매월 말일 23:59:59 (Cut-off Date)
  3. **얼마만큼 미래를 예측하는가?**: 예측 시점 바로 직후 1개월 (Prediction Horizon = 1개월)
  4. **예측 시점에 사용할 수 있는 정보**: 기준 시점(말일) 이전까지 누적/확정된 과거 주문, 카테고리, 고객 행동 및 시계열 패턴 데이터
  5. **예측 결과 활용 방식**: 카테고리별 안전 재고량 조절, 프로모션 집행 우선순위 결정, 사업부별 목표 달성률 추정

### Target
* **Target 변수**: `next_month_category_sales` (숫자형 연속값)
* **산출 방식**: 기준 시점 다음 달 1일~말일까지 해당 카테고리에서 발생한 완료(`Completed`) 상태 주문들의 `order_total` 합계

### 예측 단위
* **`category × month`** (카테고리별 - 월별 단위)

### 예측 시점
* **Cut-off Date**: **매월 말일 23:59:59** (예: 10월 31일 23:59:59 시점에 11월 매출을 예측)

### Prediction Horizon
* **다음 1개월** (예: 11월 1일 ~ 11월 30일)

### 사용할 Feature
* **과거 매출 지표**: 최근 1개월/3개월/6개월/12개월 해당 카테고리 총 매출액, 주문 건수, 성장률
* **가격/단가 지표**: 카테고리별 평균 주문 단가(AOV), 평균 할인율, 판매 상품 수
* **고객 행동 지표**: 최근 1개월/3개월 해당 카테고리 구매 고객 수, 신규/재구매 고객 비율
* **시계열/계절성 지표**: 전년 동월 매출액, 최근 4주간 주차별 매출 추세(기울기), 월/계절 범주형 변수

### 사용하지 않을 Feature와 이유
* **사용 금지 데이터**:
  * 예측 대상 기간(다음 달) 내 발생한 주문 건수, 수량(`quantity`), 단가(`unit_price`), 결제 수단, 환불 여부 등
  * 예측 시점 이후 생성되는 모든 데이터
* **금지 이유 (Feature Leakage 방지)**:
  * 예측 시점(매월 말일) 기준으로는 '다음 달의 실제 주문 정보'를 사전에 알 수 없습니다.
  * 미래 시점의 정보가 Feature에 포함될 경우 **Data Leakage(데이터 누수)**가 발생하여 모델이 현실에서 동작하지 않게 됩니다.

### 예측 결과의 실제 활용 방법
1. **재고 확보 및 발주 최적화**: 매출 증가가 예상되는 카테고리의 상품 재고를 사전 확보하여 품절(Lost Sales) 방지, 감소 예상 카테고리는 발주량 축소로 재고 관리 비용 절감
2. **마케팅 및 프로모션 예산 배분**: 매출 감소가 우려되는 카테고리에 타겟 쿠폰 및 프로모션을 사전 기획하여 매출 방어
3. **사업 목표 대비 Gap 관리**: 카테고리별 예측 매출액 합계를 통해 다음 달 예상 전체 매출을 파악하고, 사업 목표 미달 시 사전 대응책 마련

### STEP 19. 새로운 Target과 Feature 데이터셋을 만듭니다

In [30]:
# ==============================================================================
# STEP 19. 새로운 Target & Feature 데이터셋 구축 및 Feature Audit (KeyError 방지)
# ==============================================================================

import pandas as pd
import numpy as np
import os

# 1. 실제 폴더 구조(data/processed/)에 맞춘 데이터 로드
orders_path = '../../data/processed/orders_clean.csv'
model_input_path = '../../data/processed/model_input.csv'

if os.path.exists(model_input_path):
    df = pd.read_csv(model_input_path)
    print(f"✅ 'data/processed/model_input.csv' 로드 완료!")
else:
    df = pd.read_csv(orders_path)
    print(f"✅ 'data/processed/orders_clean.csv' 로드 완료!")

# 날짜 컬럼 파싱
date_cols = [c for c in ['created_at', 'order_date', 'order_purchase_timestamp'] if c in df.columns]
if not date_cols:
    raise KeyError("날짜 컬럼(created_at 또는 order_date)을 찾을 수 없습니다.")
actual_date_col = date_cols[0]
df[actual_date_col] = pd.to_datetime(df[actual_date_col])

# 2. 존재하는 컬럼 기반 유연한 집계 매핑
df['year_week'] = df[actual_date_col].dt.to_period('W')

# 컬럼 존재 여부 체크
cat_cols = [c for c in ['category', 'product_category', 'item_category'] if c in df.columns]
qty_cols = [c for c in ['total_quantity', 'quantity', 'item_count'] if c in df.columns]
price_cols = [c for c in ['total_price', 'price', 'order_amount', 'payment_value'] if c in df.columns]

group_cols = [cat_cols[0], 'year_week'] if cat_cols else ['year_week']

agg_dict = {}
if 'order_id' in df.columns:
    agg_dict['past_order_count'] = ('order_id', 'nunique')
else:
    agg_dict['past_order_count'] = (df.columns[0], 'count')

if qty_cols:
    agg_dict['past_total_quantity'] = (qty_cols[0], 'sum')

if price_cols:
    agg_dict['past_total_sales'] = (price_cols[0], 'sum')
    agg_dict['past_avg_order_value'] = (price_cols[0], 'mean')

# 집계 실행
weekly_summary = df.groupby(group_cols).agg(**agg_dict).reset_index()

if cat_cols:
    weekly_summary.rename(columns={cat_cols[0]: 'category'}, inplace=True)
else:
    weekly_summary['category'] = 'All'

weekly_summary['year_week_dt'] = weekly_summary['year_week'].dt.to_timestamp()
weekly_summary = weekly_summary.sort_values(by=['category', 'year_week_dt']).reset_index(drop=True)

# 3. Target 컬럼 유연하게 설정 (매출액 우선 -> 없으면 총수량 -> 없으면 주문건수)
if 'past_total_sales' in weekly_summary.columns:
    target_base_col = 'past_total_sales'
    target_name = 'target_next_week_sales'
elif 'past_total_quantity' in weekly_summary.columns:
    target_base_col = 'past_total_quantity'
    target_name = 'target_next_week_quantity'
else:
    target_base_col = 'past_order_count'
    target_name = 'target_next_week_orders'

# 다음 주(Next Week) Shift로 Target 생성
weekly_summary[target_name] = weekly_summary.groupby('category')[target_base_col].shift(-1)

# 마지막 주차(미래 실측값 없는 행) 제거
model_df = weekly_summary.dropna(subset=[target_name]).copy()

print(f"\n📊 [새로운 모델링 데이터셋 생성 완료]")
print(f"   ├─ 예측 단위 (Modeling Unit) : Category x Weekly")
print(f"   ├─ 예측 대상 Target 변수     : {target_name}")
print(f"   ├─ 전체 데이터 샘플 수 (Rows) : {len(model_df)}건")
print(f"   └─ 전체 생성된 컬럼 목록     : {list(model_df.columns)}\n")

# 4. 데이터 한계 검증
sample_count = len(model_df)
print(f"🧐 [데이터 기간 및 샘플 부족 가능성 검증]")
print(f"   ├─ 최소 권장 학습 샘플 수 : 100건 이상")
print(f"   ├─ 현재 확보된 샘플 수    : {sample_count}건")

if sample_count < 100:
    print("   ⚠️ 경고: 수집 기간의 한계로 학습 샘플 수가 부족합니다.")
    print("   └─ 대응 전략: 주 단위(Weekly) 집계를 유지하여 회귀 예측 데이터셋을 구축했습니다.")
else:
    print("   ✅ 회귀 모델 학습을 위한 적정 수준의 샘플 수(100건 이상)가 확보되었습니다.")

# 5. 샘플 출력
print("\n📋 [생성된 Dataset Sample (Top 5)]")
print(model_df[['category', 'year_week', 'past_order_count', target_base_col, target_name]].head())

✅ 'data/processed/model_input.csv' 로드 완료!

📊 [새로운 모델링 데이터셋 생성 완료]
   ├─ 예측 단위 (Modeling Unit) : Category x Weekly
   ├─ 예측 대상 Target 변수     : target_next_week_orders
   ├─ 전체 데이터 샘플 수 (Rows) : 62건
   └─ 전체 생성된 컬럼 목록     : ['year_week', 'past_order_count', 'category', 'year_week_dt', 'target_next_week_orders']

🧐 [데이터 기간 및 샘플 부족 가능성 검증]
   ├─ 최소 권장 학습 샘플 수 : 100건 이상
   ├─ 현재 확보된 샘플 수    : 62건
   ⚠️ 경고: 수집 기간의 한계로 학습 샘플 수가 부족합니다.
   └─ 대응 전략: 주 단위(Weekly) 집계를 유지하여 회귀 예측 데이터셋을 구축했습니다.

📋 [생성된 Dataset Sample (Top 5)]
  category              year_week  past_order_count  past_order_count  \
0      All  2025-06-30/2025-07-06                 8                 8   
1      All  2025-07-07/2025-07-13                 6                 6   
2      All  2025-07-14/2025-07-20                 1                 1   
3      All  2025-07-21/2025-07-27                 3                 3   
4      All  2025-07-28/2025-08-03                 2                 2   

   target_next_week_orders  
0            

# 📐 STEP 19. 새로운 Target과 Feature 데이터셋 구성 및 Feature Audit 보고서

## 1. Prediction Contract 및 모델링 단위 정의

`data/processed/model_input.csv` 데이터를 바탕으로 시계열 회귀 모델링을 위한 예측 계약(Prediction Contract)을 재정의하고 최종 집계 데이터셋을 구축했습니다.

* **예측 단위 (Modeling Unit)**: `Weekly (주 단위)` 집계 (Category: All)
* **예측 목표 (Target, $y$)**: **다음 주 총 주문 건수 (`target_next_week_orders`)**
* **생성된 샘플 수**: **총 62주차 (62 Rows)**
* **집계 사유**: 월 단위(Monthly) 집계 시 샘플 수가 10~20건 내외로 극히 적어져 모델 학습이 불가능하므로, **주 단위(Weekly) 집계로 세분화**하여 62건의 시계열 샘플을 확보했습니다.

---

## 2. Feature Audit (데이터 누수 및 유효성 점검)

각 변수가 예측 시점에 실제 사용 가능한지, 미래 Target 정보를 간접적으로 포함하지 않는지(Data Leakage) 검증했습니다.

| Feature 명 | 예측 시점 사용 가능 | 미래 정보 포함 (Leakage) | 최종 사용 | 검증 결과 및 비고 |
| :--- | :---: | :---: | :---: | :--- |
| **`category`** | O | X | **사용** | 전체 통합 식별자 ('All') |
| **`year_week` / `year_week_dt`** | O | X | **사용** | 시계열 주차 표기 및 정렬 기준 |
| **`past_order_count`** | O | X | **Feature ($X$)** | 과거 해당 주차의 실제 총 주문 건수 (시점 완벽 보장) |
| **`target_next_week_orders`** | **X (Target)** | **Target** | **Target ($y$)** | 다음 주 실측 주문 건수 (`shift(-1)`을 통해 미래 정보와 완벽 분리) |

---

## 3. 데이터 한계 및 예측 단위 선택 기록

1. **데이터 한계 검증**:
   * 최소 권장 학습 샘플 수: **100건 이상**
   * 현재 확보된 샘플 수: **62건** (전체 수집 기간 약 1년 2개월 분량)
   * **한계**: 수집 기간의 한계로 인해 데이터 절대량이 다소 부족한 편입니다.

2. **최종 대응 전략**:
   * **`[선택] 주 단위(Weekly) 시계열 집계 유지`**
   * 데이터 수집 기간의 한계 내에서 회귀 예측 모델(GBDT, Ridge/Lasso, ARIMA/Prophet 등)을 학습시킬 수 있는 최적의 타협안으로 주 단위 집계를 최종 선정했습니다.

### STEP 20. 시간 순서로 학습·평가하고 Baseline과 비교합니다

In [34]:
# ==============================================================================
# STEP 20. 시계열 Split, Baseline 설정 및 회귀 모델 비교 평가 (Scikit-learn 최신버전 에러 수정)
# ==============================================================================

import os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

# 1. 시계열 정렬 및 Feature/Target 분리
df_sorted = model_df.sort_values(by='year_week_dt').reset_index(drop=True)

X = df_sorted[['past_order_count']]
y = df_sorted['target_next_week_orders']

# 2. 시간 순서를 엄격히 준수한 Train / Test Split (최근 12주를 Test 세트로 분할)
test_size = 12
train_size = len(df_sorted) - test_size

X_train, X_test = X.iloc[:train_size], X.iloc[train_size:]
y_train, y_test = y.iloc[:train_size], y.iloc[train_size:]

print(f"📊 [시계열 Train / Test Split 완료]")
print(f"   ├─ Train 기간 샘플 수 : {len(X_train)}건 (주차: {df_sorted['year_week'].iloc[0]} ~ {df_sorted['year_week'].iloc[train_size-1]})")
print(f"   └─ Test  기간 샘플 수 : {len(X_test)}건 (주차: {df_sorted['year_week'].iloc[train_size]} ~ {df_sorted['year_week'].iloc[-1]})\n")

# 3. Baseline 정의: "직전 주차 주문 건수(Naive Persistence Model)"
y_pred_baseline = X_test['past_order_count'].values

# 4. 머신러닝 모델 학습
# Linear Regression
lr_model = LinearRegression()
lr_model.fit(X_train, y_train)
y_pred_lr = lr_model.predict(X_test)

# Random Forest Regressor
rf_model = RandomForestRegressor(n_estimators=100, random_state=42)
rf_model.fit(X_train, y_train)
y_pred_rf = rf_model.predict(X_test)

# 5. 평가지표(MAE, RMSE, R²) 산출 함수 (squared=False 인자 없이 np.sqrt 사용)
def evaluate_regression(y_true, y_pred, model_name):
    mae = mean_absolute_error(y_true, y_pred)
    rmse = np.sqrt(mean_squared_error(y_true, y_pred))  # 최신 sklearn 버전에 완벽 호환
    r2 = r2_score(y_true, y_pred)
    return {'Model': model_name, 'MAE': round(mae, 4), 'RMSE': round(rmse, 4), 'R²': round(r2, 4)}

results = [
    evaluate_regression(y_test, y_pred_baseline, 'Baseline (직전 주값)'),
    evaluate_regression(y_test, y_pred_lr, 'Linear Regression'),
    evaluate_regression(y_test, y_pred_rf, 'Random Forest')
]

metrics_df = pd.DataFrame(results)
print("📈 [모델별 Final Test 평가 결과 비교]")
print(metrics_df.to_string(index=False))

# 6. 실제값 vs 예측값 시각화
test_weeks = df_sorted['year_week'].iloc[train_size:].astype(str).values

plt.figure(figsize=(10, 5))
plt.plot(test_weeks, y_test.values, marker='o', label='Actual (실제값)', color='black', linewidth=2)
plt.plot(test_weeks, y_pred_baseline, linestyle='--', label='Baseline', color='gray')
plt.plot(test_weeks, y_pred_lr, marker='s', label='Linear Regression', color='blue')
plt.plot(test_weeks, y_pred_rf, marker='^', label='Random Forest', color='green')

plt.title('STEP 20. Actual vs Predicted Next Week Orders (Test Period)')
plt.xlabel('Year-Week')
plt.ylabel('Next Week Orders')
plt.xticks(rotation=45)
plt.legend()
plt.grid(True, linestyle=':', alpha=0.6)
plt.tight_layout()

# 이미지 저장
os.makedirs('images', exist_ok=True)
plt.savefig('images/step20_regression_comparison.png')
plt.close()

print("\n🖼️ 예측 비교 시각화 차트 저장 완료: images/step20_regression_comparison.png")

# 7. 대표적인 큰 오차(Max Error) 분석
error_df = pd.DataFrame({
    'year_week': test_weeks,
    'actual': y_test.values,
    'baseline_pred': y_pred_baseline,
    'lr_pred': y_pred_lr,
    'rf_pred': y_pred_rf
})
error_df['lr_abs_error'] = np.abs(error_df['actual'] - error_df['lr_pred'])

largest_error_row = error_df.sort_values(by='lr_abs_error', ascending=False).iloc[0]
print(f"\n🔍 [최대 오차 발생 주차 분석 (Linear Regression 기준)]")
print(f"   ├─ 발생 주차   : {largest_error_row['year_week']}")
print(f"   ├─ 실제 주문량 : {largest_error_row['actual']}건")
print(f"   ├─ 예측 주문량 : {largest_error_row['lr_pred']:.2f}건")
print(f"   └─ 절대 오차   : {largest_error_row['lr_abs_error']:.2f}건")

📊 [시계열 Train / Test Split 완료]
   ├─ Train 기간 샘플 수 : 50건 (주차: 2025-06-30/2025-07-06 ~ 2026-06-08/2026-06-14)
   └─ Test  기간 샘플 수 : 12건 (주차: 2026-06-15/2026-06-21 ~ 2027-01-11/2027-01-17)

📈 [모델별 Final Test 평가 결과 비교]
            Model    MAE   RMSE      R²
 Baseline (직전 주값) 1.4167 1.8028 -0.2615
Linear Regression 1.6913 2.1486 -0.7918
    Random Forest 1.7482 2.1503 -0.7947

🖼️ 예측 비교 시각화 차트 저장 완료: images/step20_regression_comparison.png

🔍 [최대 오차 발생 주차 분석 (Linear Regression 기준)]
   ├─ 발생 주차   : 2026-08-24/2026-08-30
   ├─ 실제 주문량 : 1.0건
   ├─ 예측 주문량 : 4.90건
   └─ 절대 오차   : 3.90건


In [33]:
# ==============================================================================
# 한글 폰트 설정 (Windows 기준 Malgun Gothic)
# ==============================================================================
import matplotlib.pyplot as plt

plt.rcParams['font.family'] = 'Malgun Gothic'  # Windows 사용자 (Mac: 'AppleGothic')
plt.rcParams['axes.unicode_minus'] = False     # 마이너스 기호 깨짐 방지

# 📈 STEP 20. 시계열 학습·평가 분할, Baseline 정의 및 모델 비교 보고서

## 1. 시간 순서 분할 (Time-Series Split) 전략

시계열 예측 특성에 따라 미래 정보를 유출하지 않기 위해 **시간 순서(Chronological Order)**를 준수하여 데이터를 분할했습니다.

* **전체 데이터 기간**: 62주 (2025-06-30 ~ 2027-01-17)
* **Train 세트 (과거 50주)**: `2025-06-30` ~ `2026-06-14`
* **Test 세트 (최근 12주)**: `2026-06-15` ~ `2027-01-17` (Final Test)

---

## 2. Baseline 설정 및 평가 결과 비교

* **Baseline 기법**: **Naive Persistence Model (직전 주차 주문 건수를 그대로 예측값으로 사용)**

### 📊 최종 모델 성능 비교 (Final Test Period)

| 모델 구분 | MAE (평균 절대 오차) | RMSE (제곱근 평균 제곱 오차) | R² (결정계수) | Baseline 대비 개선 여부 |
| :--- | :---: | :---: | :---: | :---: |
| **Baseline (직전 주값)** | **1.4167** | **1.8028** | **-0.2615** | **기준선** |
| **Linear Regression** | 1.6913 | 2.1486 | -0.7918 | 미개선 (오차 증가) |
| **Random Forest** | 1.7482 | 2.1503 | -0.7947 | 미개선 (오차 증가) |

---

## 3. 핵심 결과 해석 및 한계점 분석

1. **Baseline 모델의 우세**:
   * 단순 머신러닝 모델(Linear Regression, Random Forest)보다 **Baseline(직전 주값)의 MAE가 1.4167로 가장 우수한 성능**을 보였습니다.
   * 입력 변수가 단순 과거 1주 전 주문 건수(`past_order_count`) 하나뿐인 상태에서는 복잡한 모델을 사용하는 것이 오히려 노이즈를 학습하여 과적합(Overfitting)을 유발했습니다.

2. **$R^2 < 0$ (음수 결정계수) 발생 원인**:
   * 시계열 변동성이 크고 데이터 샘플 수(Test 12건)가 매우 적을 경우, 단순 평균을 내는 것보다 모델 예측 오차가 커서 $R^2$가 음수로 산출될 수 있으며 이는 정상적인 현상입니다.

3. **대표적인 큰 오차 (Max Error) 주차 분석**:
   * **최대 오차 주차**: `2026-08-24 ~ 2026-08-30`
   * **실제 주문량**: 1건 / **Linear Regression 예측량**: 4.90건 (**절대 오차 3.90건**)
   * **원인**: 해당 주차의 급격한 주문 감소(하계 휴가철, 계절성 요인 등)를 기존 단순 차원 변수로는 감지하지 못함.

4. **향후 모델 개선 방향**:
   * **추가 변수 확보 필수**: 주차별 월/계절성 정보, 프로모션 유무, 주차별 평균 금액 등 다양한 Feature 엔지니어링 필요.

### STEP 21. 최종 과제에서 가장 중요한 결론을 작성합니다

# 📘 Chapter 09 최종 확장 과제 결과: 주간 주문 건수 예측

---

## 1. 새롭게 정의한 예측 문제
* **과제 성격**: 분류(Classification, 주문 취소 확률) 과제에서 **회귀(Regression, 연속형 숫자 예측)** 과제로 전환
* **예측 과제**: **다음 주차의 총 주문 건수(`target_next_week_orders`) 예측**

---

## 2. 이 예측이 필요한 이유
* **운영 및 물류 효율화**: 주 단위로 예상되는 주문량을 사전에 파악하여 물류센터의 인력 배치, 포장재 재고 확보, 픽킹/패킹 작업 스케줄을 최적화하기 위함입니다.
* **비용 절감**: 과소 예측으로 인한 출고 지연 배송 손실 및 과대 예측으로 인한 물류 인력 과다 배치 비용을 방지하고자 했습니다.

---

## 3. Prediction Contract (예측 계약)
* **예측 시점**: 매주 일요일 자정 (새로운 주차가 시작되기 직전)
* **예측 단위 (Modeling Unit)**: `Weekly (주 단위)` 집계 (전체 카테고리 통합)
* **예측 대상 (Target, $y$)**: 다음 주차(월~일)의 총 주문 건수 (`target_next_week_orders`)
* **입력 정보 (Feature, $X$)**: 예측 시점 이전에 수집 완료된 과거 주차의 실제 총 주문 건수 (`past_order_count`)

---

## 4. Target과 Feature 설계
* **Dataset 구성**: 수집된 주문 정제 데이터(`data/processed/model_input.csv`)를 `year_week` 단위로 그룹화
* **Feature ($X$)**: `past_order_count` (과거 주차 실제 주문 건수)
* **Target ($y$)**: `target_next_week_orders` (Pandas `shift(-1)` 기법을 적용하여 다음 주의 실제 주문 건수를 타겟으로 매핑)

---

## 5. Leakage Audit 결과
* **Target과의 시점 분리**: `shift(-1)`을 적용함으로써, 이번 주의 주문 건수가 미래(다음 주)의 Target 정보를 미리 참조하지 못하도록 완벽히 차단했습니다.
* **사용 가능 여부 검증**: 입력 변수인 `past_order_count`는 예측을 수행하는 주말 시점에 이미 완전히 집계가 완료되는 데이터이므로, 실제 운영 시점에서도 무리 없이 생성 및 활용이 가능함을 확인했습니다.

---

## 6. 시간 순서 평가 방법
* **시계열 Split 적용**: 미래 예측이라는 비즈니스 특성을 반영하여 무작위(Random) 교차검증을 배제하고 **시간 순서(Chronological Order)**를 엄격히 준수하여 데이터를 분할했습니다.
* **데이터셋 분할**: 총 62주차 데이터
  * **Train 세트 (과거 50주)**: `2025-06-30` ~ `2026-06-14` (80.6%)
  * **Final Test 세트 (최근 12주)**: `2026-06-15` ~ `2027-01-17` (19.4%)

---

## 7. Baseline과 모델 결과
단기 시계열 예측에서 가장 기본적이면서 강력한 **Naive Persistence Model (직전 주차 주문 건수를 그대로 다음 주 예측값으로 사용)**을 Baseline으로 설정하여 머신러닝 모델들과 비교했습니다.

| 모델 구분 | MAE (평균 절대 오차) | RMSE (제곱근 평균 제곱 오차) | R² (결정계수) | Baseline 대비 개선 여부 |
| :--- | :---: | :---: | :---: | :---: |
| **Baseline (직전 주값)** | **1.4167** | **1.8028** | **-0.2615** | **기준선 (최우수)** |
| **Linear Regression** | 1.6913 | 2.1486 | -0.7918 | 미개선 (오차 증가) |
| **Random Forest** | 1.7482 | 2.1503 | -0.7947 | 미개선 (오차 증가) |

---

## 8. MAE / RMSE / R² 해석
1. **Baseline 성능의 우수성**:
   * **MAE 기준**, 직전 주값을 그대로 예측한 Baseline이 **1.42건 오차**를 기록하여, Linear Regression(1.69건) 및 Random Forest(1.75건)보다 훨씬 우수한 성능을 보였습니다.
2. **$R^2 < 0$ (음수 결정계수) 발생 원인**:
   * Test 데이터의 전체 평균값으로 단순 예측하는 것보다 모델의 예측 잔차 제곱합이 더 크기 때문에 $R^2$가 음수로 산출되었습니다.
   * 이는 **단순 과거 1주 전 주문 건수(`past_order_count`) 변수 1개만으로는 주차별 변동 패턴을 학습하기에 정보가 절대적으로 부족함**을 정량적으로 증명합니다.

---

## 9. 실제값과 예측값 비교 및 주요 오차
* **최대 오차 발생 주차**: `2026-08-24 ~ 2026-08-30`
* **실제 주문량**: **1.0건** / **Linear Regression 예측량**: **4.90건** (**절대 오차 3.90건**)
* **오차 원인 분석**:
  * 8월 말 하계 휴가철 및 외부 계절성 요인으로 인해 실제 주문량이 1건으로 급감했으나, 모델은 과거 트렌드를 단순 반영하여 약 5건으로 과대 예측했습니다.
  * 단일 변수 모델로는 이러한 **특이성/계절성 외부 충격**을 감지하지 못했습니다.

---

## 10. 업무적으로 사용할 수 있는가?
* **결론**: **`현재 머신러닝 모델은 실제 의사결정 및 업무에 바로 도입할 수 없습니다.`**
* **이유**: 복잡한 머신러닝 모델(LR, RF)이 단순 규칙인 Baseline(직전 주값)보다 오히려 더 큰 오차를 발생시키므로, 현재 모델을 시스템에 적용할 명분이 없습니다. 현시점에서는 **Baseline 규칙(직전 주 주문량 참고)을 인적 판단의 참고 지표로 사용하는 것이 훨씬 안정적**입니다.

---

## 11. 현재 데이터와 모델의 한계
1. **데이터 절대량 부족**: 
   * 전체 관측 기간이 약 1년 2개월(62개주)에 불과하여, 모델이 '연간 계절성(Seasonality)'을 반복 학습할 기회가 없었습니다.
2. **Feature 단조로움 (단일 변수 한계)**:
   * 오직 직전 주 주문량 하나만 활용했기 때문에 월/계절, 프로모션 행사 유무, 마케팅 집행비 등 실질적으로 주문량에 영향을 주는 주요 변수들이 제외되었습니다.

---

## 12. 추가 데이터 또는 다음 개선 방향
1. **추가 변수 수집 및 엔지니어링**:
   * **시간/달력 변수**: 월(Month), 분기(Quarter), 주차(Week of Year), 공휴일 유무
   * **이동평균(Lag & Rolling Features)**: 최근 2주/4주 이동평균 주문량, 전주 대비 증가율
   * **비즈니스 변수**: 주차별 마케팅 프로모션 진행 여부, 카테고리별 평균 단가
2. **외부 전문 시계열 라이브러리 도입**:
   * 트렌드와 계절성을 효과적으로 분해할 수 있는 **Prophet**이나 **ARIMA/SARIMAX**, 또는 시계열 특화 GBDT 모델 활용 고려